# Notebook 6 — Mini-VLA: CLIP + MobileViT-Small + PerceiverResampler + Action Chunking

| Component | Choice |
|---|---|
| Vision Encoder | `MobileViT-Small (embed_dim=128)` — MBConv stages + patch-level transformer blocks |
| Text Encoder | `CLIP Text (frozen, 512-dim)` |
| Token Learner | `PerceiverResampler (32 latents, depth=2, cross+self attention)` |
| Temporal Fusion | `CausalTemporalAttention (causal self-attn over frames)` |
| Transformer | `4-layer, 8-head, pre-norm` |
| Special | `Action Chunking — predicts CHUNK_SIZE=4 future actions simultaneously` |

> XAI outputs saved to `xai_outputs_{MODEL_NAME}/`


In [1]:
# ========================= BLOCK 1: IMPORTS =========================
import sys, os, gc, time, math, random, warnings, subprocess
import numpy as np
from collections import Counter, defaultdict

# ─── CRITICAL: Restrict TensorFlow GPU memory BEFORE importing TF ────────
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from einops import rearrange, reduce, repeat
from einops.layers.torch import Rearrange, Reduce
from transformers import (
    DistilBertModel, DistilBertTokenizer,
    BertModel, BertTokenizer,
    CLIPTokenizer, CLIPTextModel,
)
from tqdm.auto import tqdm
from PIL import Image as PILImage

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import tensorflow as tf
# ─── Limit TF to minimal GPU memory ──────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    # Optionally hard-limit TF to 512 MB so PyTorch gets the rest
    try:
        tf.config.set_logical_device_configuration(
            gpus[0],
            [tf.config.LogicalDeviceConfiguration(memory_limit=512)])
    except RuntimeError:
        pass  # Already initialized
import tensorflow_datasets as tfds

warnings.filterwarnings('ignore')

# H100 / RTX optimizations
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
cudnn.benchmark = True
torch.set_float32_matmul_precision('high')
torch.set_num_threads(8)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU:  {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")
print("✅ Block 1 done")

2026-03-04 00:04:54.788395: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-04 00:04:54.833393: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-04 00:04:56.165251: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Device: cuda
  GPU:  NVIDIA GeForce RTX 3060 Laptop GPU
  VRAM: 6 GB
✅ Block 1 done


In [2]:
# ========================= BLOCK 2: CONFIGURATION =========================
SEED=42; torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
if DEVICE=="cuda": torch.cuda.manual_seed_all(SEED)

TARGET_SIZE=224; SEQ_LEN=6; SPLIT_RATIO=0.8; ACTION_BINS=256
# MobileViT-Small spatial flow:
#   Stem(stride=2)→112×112  DS1→56×56  DS2→28×28  proj→28×28  embed_dim=128
MOBILEVIT_DROP_PATH=0.15
CHUNK_SIZE=4              # action chunking: predict 4 future timesteps at once
NUM_VISUAL_TOKENS=32      # PerceiverResampler latents
TRANSFORMER_DEPTH=4; TRANSFORMER_HEADS=8
DROPOUT=0.2; DROP_PATH_RATE=0.15
BATCH_SIZE=1; GRADIENT_ACCUM_STEPS=128; EPOCHS=80; LR=3e-5; WEIGHT_DECAY=0.15
LION_BETAS=(0.9,0.99); WARMUP_STEPS=500; GRAD_CLIP=1.0
LABEL_SMOOTHING=0.1; MIXUP_ALPHA=0.3; USE_CUTMIX=True; FOCAL_GAMMA=2.5; PATIENCE=10
NUM_WORKERS=2; PREFETCH_FACTOR=4; USE_BFLOAT16=True; AUTOCAST_DTYPE=torch.bfloat16
TEXT_DIM=512              # CLIP text encoder output dimension
MODEL_NAME="mini_vla_mobilevit"; VIS_ENC_NAME="MobileViT-Small"
TEXT_ENC_NAME="CLIP"; TL_NAME="PerceiverResampler"
print("✅ Block 2 done")
print(f"   CHUNK_SIZE={CHUNK_SIZE}  NUM_VISUAL_TOKENS={NUM_VISUAL_TOKENS}  TEXT_DIM={TEXT_DIM}")


✅ Block 2 done
   CHUNK_SIZE=4  NUM_VISUAL_TOKENS=32  TEXT_DIM=512


In [3]:
# ========================= BLOCK 3: DOWNLOAD DATASET =========================
DATASET_NAME = "berkeley_cable_routing"
DATA_DIR     = os.path.join(os.getcwd(), "berkeley_cable_routing_data")

def check_existing():
    if not os.path.exists(DATA_DIR): return False
    for root, dirs, files in os.walk(DATA_DIR):
        if "dataset_info.json" in files:
            total = sum(os.path.getsize(os.path.join(root,f)) for f in files)/1e9
            print(f"✅ Already downloaded: {root} ({total:.2f} GB)"); return True
    return False

def download_gcs_tf():
    src = f"gs://gresearch/robotics/{DATASET_NAME}"
    print(f"📥 Downloading from {src}")
    def list_all(d):
        out=[]
        for item in tf.io.gfile.listdir(d):
            c=item.rstrip('/');
            if '$folder$' in c: continue
            full=f"{d}/{c}"
            if tf.io.gfile.isdir(full): out.extend(list_all(full))
            else: out.append(full)
        return out
    files=list_all(src); print(f"   Found {len(files)} files")
    for i,p in enumerate(files):
        rel=p[len(src)+1:]; local=os.path.join(DATA_DIR,rel)
        os.makedirs(os.path.dirname(local),exist_ok=True)
        if os.path.exists(local):
            try:
                if os.path.getsize(local)==tf.io.gfile.stat(p).length: continue
            except: pass
        tf.io.gfile.copy(p,local,overwrite=True)
        print(f"   [{i+1}/{len(files)}] {rel}")
    print("✅ Download complete")

if not check_existing(): download_gcs_tf()

BUILDER_DIR=None
for root,dirs,files in os.walk(DATA_DIR):
    if "dataset_info.json" in files: BUILDER_DIR=root; break
if BUILDER_DIR is None: raise FileNotFoundError(f"dataset_info.json not in {DATA_DIR}")

# Force builder creation on CPU
with tf.device('/CPU:0'):
    builder = tfds.builder_from_directory(builder_dir=BUILDER_DIR)
for s,si in builder.info.splits.items(): print(f"   {s}: {si.num_examples:,} episodes")
print("✅ Block 3 done")

✅ Already downloaded: /media/sparsh/CaptainSlow/Programming Stuff/Programming Stuff/Code _n_ Stuff/All Projects/All Projects/VLAs/my work/berkeley_cable_routing_data/0.1.0 (5.01 GB)
   train: 1,482 episodes
   test: 165 episodes
✅ Block 3 done


2026-03-04 00:04:57.137169: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1772562897.137188 1137921 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1772562897.138076 1137921 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 512 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [4]:
# ========================= BLOCK 4: EXPLORE STRUCTURE =========================
# Force TF operations onto CPU to avoid VRAM usage
with tf.device('/CPU:0'):
    ds_peek = builder.as_dataset(split="train[:1]")
    ep=next(iter(ds_peek)); steps=list(ep["steps"]); step=steps[0]
    obs,act=step["observation"],step["action"]

    IMG_KEY=None
    for k,v in obs.items():
        if hasattr(v,'shape') and len(v.shape)==3 and v.shape[-1]==3:
            IMG_KEY=k; IMG_SHAPE=tuple(v.shape)

    ACT_KEYS=sorted(act.keys())
    sample_action=np.concatenate([np.atleast_1d(act[k].numpy()).astype(np.float32) for k in ACT_KEYS])
    ACTUAL_ACTION_DIM=len(sample_action); NUM_ACTIONS=ACTUAL_ACTION_DIM

    INST_KEY=None
    for k,v in obs.items():
        val=v.numpy() if hasattr(v,'numpy') else v
        if isinstance(val,bytes): INST_KEY=k; break
        if isinstance(val,np.ndarray) and val.dtype.kind in ('U','S','O'): INST_KEY=k; break

print(f"Image: '{IMG_KEY}' {IMG_SHAPE}")
print(f"Action: {ACT_KEYS}, dim={ACTUAL_ACTION_DIM}")
print(f"Instruction: '{INST_KEY}'")
del ds_peek,ep,steps; gc.collect()
print("✅ Block 4 done")

2026-03-04 00:04:57.289427: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
2026-03-04 00:04:57.383766: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Image: 'wrist45_image' (128, 128, 3)
Action: ['rotation_delta', 'terminate_episode', 'world_vector'], dim=7
Instruction: 'natural_language_instruction'
✅ Block 4 done


In [5]:
# ========================= BLOCK 5: UTILITIES =========================
class RT1Tokenizer:
    def __init__(self, vocab_size=256, range_min=-3.2, range_max=3.2):
        self.vocab_size=vocab_size; self.min=range_min; self.max=range_max
    def encode(self, af):
        af=torch.tensor(af,dtype=torch.float32) if not isinstance(af,torch.Tensor) else af.float()
        af=torch.clamp(af,self.min,self.max)
        return ((af-self.min)/(self.max-self.min)*(self.vocab_size-1)).long()
    def decode(self, t):
        return t.float()/(self.vocab_size-1)*(self.max-self.min)+self.min

class DropPath(nn.Module):
    def __init__(self, p=0.0): super().__init__(); self.p=p
    def forward(self, x):
        if self.p==0. or not self.training: return x
        keep=1-self.p; shape=(x.shape[0],)+(1,)*(x.ndim-1)
        mask=(torch.rand(shape,dtype=x.dtype,device=x.device)+keep).floor_()
        return x/keep*mask

print("✅ Block 5 done")


✅ Block 5 done


In [6]:
# ========================= BLOCK 6: MODEL COMPONENTS =========================

# ─── Shared primitives ───────────────────────────────────────────────────────
class ImprovedSqueezeExcitation(nn.Module):
    def __init__(self, dim, reduction_ratio=4):
        super().__init__()
        rd=max(1,dim//reduction_ratio)
        self.se=nn.Sequential(
            Reduce('b c h w -> b c','mean'),
            nn.Linear(dim,rd,bias=False),nn.BatchNorm1d(rd),nn.SiLU(),nn.Dropout(0.1),
            nn.Linear(rd,dim,bias=False),nn.Sigmoid(),Rearrange('b c -> b c 1 1'))
    def forward(self,x): return x*self.se(x)

class ImprovedMBConv(nn.Module):
    def __init__(self,dim_in,dim_out,*,downsample,expansion_rate=4,use_fused=False,
                 drop_path_rate=0.1,se_ratio=0.25):
        super().__init__()
        self.use_residual=(dim_in==dim_out) and (not downsample)
        hidden=int(expansion_rate*dim_in); stride=2 if downsample else 1; se_rd=int(1/se_ratio)
        if use_fused:
            self.net=nn.Sequential(
                nn.Conv2d(dim_in,hidden,3,stride=stride,padding=1,bias=False),
                nn.BatchNorm2d(hidden),nn.SiLU(),
                ImprovedSqueezeExcitation(hidden,se_rd),
                nn.Conv2d(hidden,dim_out,1,bias=False),nn.BatchNorm2d(dim_out))
        else:
            self.net=nn.Sequential(
                nn.Conv2d(dim_in,hidden,1,bias=False),nn.BatchNorm2d(hidden),nn.SiLU(),
                nn.Conv2d(hidden,hidden,3,stride=stride,padding=1,groups=hidden,bias=False),
                nn.BatchNorm2d(hidden),nn.SiLU(),
                ImprovedSqueezeExcitation(hidden,se_rd),
                nn.Conv2d(hidden,dim_out,1,bias=False),nn.BatchNorm2d(dim_out))
        from torch.nn import Identity
        self.drop_path=DropPath(drop_path_rate) if (drop_path_rate>0 and self.use_residual) else nn.Identity()
    def forward(self,x): return x+self.drop_path(self.net(x)) if self.use_residual else self.net(x)

# ─── MobileViT building block ────────────────────────────────────────────────
class MobileViTBlock(nn.Module):
    """
    Hybrid local-global block from MobileViT (Mehta & Rastegari 2021).
    1. Local branch : depthwise 3×3 + pointwise 1×1 (captures local textures)
    2. Global branch: non-overlapping p×p patches → Transformer → fold back
                      (captures long-range pixel correlations without positional bias)
    3. Fusion       : concat(local, global) → 1×1 conv → residual add
    """
    def __init__(self, dim, depth=2, heads=4, patch_size=2, mlp_ratio=2, dropout=0.1):
        super().__init__()
        self.p = patch_size
        # Local representation
        self.local_rep = nn.Sequential(
            nn.Conv2d(dim, dim, 3, padding=1, groups=dim, bias=False),
            nn.BatchNorm2d(dim), nn.SiLU(),
            nn.Conv2d(dim, dim, 1, bias=False))
        # Patch-level layer norm before global transformer
        self.patch_norm = nn.LayerNorm(dim)
        # Global transformer (operates on p×p patch tokens)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=dim*mlp_ratio,
            dropout=dropout, batch_first=True, norm_first=True, activation='gelu')
        self.transformer = nn.TransformerEncoder(enc_layer, depth)
        # Fusion
        self.fusion = nn.Sequential(
            nn.Conv2d(2*dim, dim, 1, bias=False),
            nn.BatchNorm2d(dim))

    def forward(self, x):
        B, C, H, W = x.shape; p = self.p
        assert H % p == 0 and W % p == 0, f"H={H}, W={W} must be divisible by patch={p}"
        nh, nw = H // p, W // p
        # ── Local ──
        local_out = self.local_rep(x)                        # (B, C, H, W)
        # ── Global ── unfold into (B*nh*nw, p², C) patches
        xr = x.reshape(B, C, nh, p, nw, p)                  # (B, C, nh, p, nw, p)
        xr = xr.permute(0, 2, 4, 3, 5, 1)                   # (B, nh, nw, p, p, C)
        xr = xr.reshape(B * nh * nw, p * p, C)              # (B*nh*nw, p², C)
        xr = self.patch_norm(xr)
        xr = self.transformer(xr)                            # (B*nh*nw, p², C)
        # fold back
        xr = xr.reshape(B, nh, nw, p, p, C)
        xr = xr.permute(0, 5, 1, 3, 2, 4)                   # (B, C, nh, p, nw, p)
        global_out = xr.reshape(B, C, H, W)                  # (B, C, H, W)
        # ── Fuse & residual ──
        return x + self.fusion(torch.cat([local_out, global_out], dim=1))

# ─── MobileViT-Small encoder ─────────────────────────────────────────────────
class MobileViTEncoder(nn.Module):
    """
    MobileViT-Small spatial flow (224×224 input):
      Stem   : 3→32, stride=2              →  112×112, 32ch
      Stage-0: MBConv×2  (32→32, stride=1) →  112×112, 32ch
      DS-1   : MBConv    (32→64, stride=2) →   56×56,  64ch
      Stage-1: MBConv×2 + MobileViTBlock   →   56×56,  64ch  (local+global)
      DS-2   : MBConv    (64→96, stride=2) →   28×28,  96ch
      Stage-2: MobileViTBlock×2            →   28×28,  96ch  (deep global)
      Proj   : Conv1×1   (96→128)          →   28×28, 128ch
    embed_dim = 128
    """
    def __init__(self, drop_path_rate=0.15):
        super().__init__()
        dp = drop_path_rate
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.SiLU())

        self.stage0 = nn.Sequential(
            ImprovedMBConv(32, 32, downsample=False, use_fused=True, drop_path_rate=dp*0.3),
            ImprovedMBConv(32, 32, downsample=False, use_fused=True, drop_path_rate=dp*0.5))
        self.ds1 = ImprovedMBConv(32, 64, downsample=True, use_fused=True, drop_path_rate=dp*0.6)

        self.stage1 = nn.Sequential(
            ImprovedMBConv(64, 64, downsample=False, use_fused=False, drop_path_rate=dp*0.7),
            ImprovedMBConv(64, 64, downsample=False, use_fused=False, drop_path_rate=dp*0.8),
            MobileViTBlock(64, depth=2, heads=4, patch_size=2))
        self.ds2 = ImprovedMBConv(64, 96, downsample=True, use_fused=False, drop_path_rate=dp*0.9)

        self.stage2 = nn.Sequential(
            MobileViTBlock(96, depth=3, heads=8, patch_size=2),
            MobileViTBlock(96, depth=2, heads=8, patch_size=2))

        self.proj = nn.Sequential(
            nn.Conv2d(96, 128, 1, bias=False),
            nn.BatchNorm2d(128), nn.SiLU())
        self.embed_dim = 128
        # ModuleList alias so _get_target_layer can walk stages
        self.stages = nn.ModuleList([self.stage0, self.stage1, self.stage2])

    def forward(self, x):
        x = self.stem(x)    # (B, 32, 112, 112)
        x = self.stage0(x)  # (B, 32, 112, 112)
        x = self.ds1(x)     # (B, 64,  56,  56)
        x = self.stage1(x)  # (B, 64,  56,  56)
        x = self.ds2(x)     # (B, 96,  28,  28)
        x = self.stage2(x)  # (B, 96,  28,  28)
        x = self.proj(x)    # (B,128,  28,  28)
        return x

# ─── Perceiver Resampler (Token Learner) ─────────────────────────────────────
class PerceiverResampler(nn.Module):
    """
    Perceiver-style token compression (Jaegle et al. 2021; Alayrac et al. Flamingo 2022).
    Learns a fixed set of latent queries that cross-attend to spatial visual features,
    compressing arbitrary spatial maps into a fixed number of rich tokens.

    Input : (B, C, H, W)
    Output: (B, num_latents, C)
    """
    def __init__(self, dim, num_latents=32, depth=2, heads=4, dropout=0.1):
        super().__init__()
        self.latents = nn.Parameter(torch.randn(1, num_latents, dim) * 0.02)
        self.layers = nn.ModuleList()
        for _ in range(depth):
            self.layers.append(nn.ModuleDict({
                'cross_attn': nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True),
                'cross_norm': nn.LayerNorm(dim),
                'self_attn' : nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True),
                'self_norm' : nn.LayerNorm(dim),
                'ffn'       : nn.Sequential(
                                  nn.LayerNorm(dim),
                                  nn.Linear(dim, dim*4), nn.GELU(), nn.Dropout(dropout),
                                  nn.Linear(dim*4, dim), nn.Dropout(dropout)),
            }))
        self.out_norm = nn.LayerNorm(dim)

    def forward(self, x):
        B, C, H, W = x.shape
        kv = x.reshape(B, C, H*W).permute(0, 2, 1)    # (B, H*W, C) — keys/values
        q  = self.latents.expand(B, -1, -1)            # (B, num_latents, C) — queries
        for layer in self.layers:
            ca, _ = layer['cross_attn'](q, kv, kv)    # latents attend to spatial
            q = layer['cross_norm'](q + ca)
            sa, _ = layer['self_attn'](q, q, q)        # latents refine among themselves
            q = layer['self_norm'](q + sa)
            q = q + layer['ffn'](q)                    # FFN
        return self.out_norm(q)                        # (B, num_latents, C)

# ─── Causal Temporal Attention ────────────────────────────────────────────────
class CausalTemporalAttention(nn.Module):
    """
    Causal self-attention over the temporal (frame) dimension.
    Each frame may only attend to itself and past frames (autoregressive inductive bias),
    encouraging the model to capture temporal dynamics for action prediction.

    Input : (B, D, F, H, W)
    Output: (B, D, F, H, W)  [residual — spatial dims unchanged]
    """
    def __init__(self, dim, heads=4, dropout=0.1):
        super().__init__()
        self.pool  = nn.AdaptiveAvgPool3d((None, 1, 1))    # pool H,W → scalar per frame
        self.attn  = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm  = nn.LayerNorm(dim)
        self.proj  = nn.Sequential(nn.Linear(dim, dim*2), nn.GELU(), nn.Linear(dim*2, dim))
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        B, D, F, H, W = x.shape
        # Temporal summary: pool spatial → (B, D, F)
        pooled = self.pool(x).squeeze(-1).squeeze(-1)      # (B, D, F)
        pooled = pooled.permute(0, 2, 1)                   # (B, F, D)
        # Causal mask: frame i cannot attend to frame j > i
        mask = torch.triu(torch.ones(F, F, device=x.device), diagonal=1).bool()
        attn_out, _ = self.attn(pooled, pooled, pooled, attn_mask=mask)
        pooled = self.norm(pooled + self.drop(attn_out))   # (B, F, D)
        delta  = self.proj(pooled)                         # (B, F, D)
        # Broadcast residual back to full spatial tensor
        delta = delta.permute(0, 2, 1).unsqueeze(-1).unsqueeze(-1)   # (B, D, F, 1, 1)
        return x + delta.expand_as(x)

# ─── Text Encoders (all kept for completeness) ────────────────────────────────
class DistilBERTEncoder:
    def __init__(self,device='cuda',freeze=True):
        self.device=device; self.frozen=freeze
        self.tokenizer=DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
        self.model=DistilBertModel.from_pretrained('distilbert-base-uncased').to(device)
        self.embed_dim=768
        if freeze: [p.__setattr__('requires_grad',False) for p in self.model.parameters()]; self.model.eval()
        print(f"   ✅ DistilBERT (768-dim, frozen={freeze})")
    def encode(self,instructions):
        if isinstance(instructions,str): instructions=[instructions]
        enc=self.tokenizer(instructions,padding=True,truncation=True,max_length=77,return_tensors='pt').to(self.device)
        ctx=torch.no_grad() if self.frozen else torch.enable_grad()
        with ctx: out=self.model(**enc)
        return out.last_hidden_state[:,0,:]

class BERTEncoder:
    def __init__(self,device='cuda',freeze=True,unfreeze_top_k=0):
        self.device=device; self.frozen=freeze
        self.tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')
        self.model=BertModel.from_pretrained('bert-base-uncased').to(device)
        self.embed_dim=768
        if freeze:
            for p in self.model.parameters(): p.requires_grad=False
            if unfreeze_top_k>0:
                n=len(self.model.encoder.layer)
                for layer in self.model.encoder.layer[n-unfreeze_top_k:]:
                    for p in layer.parameters(): p.requires_grad=True
                self.frozen=f'partial(top-{unfreeze_top_k})'
            else: self.model.eval()
        print(f"   ✅ BERT (768-dim, frozen={self.frozen})")
    def encode(self,instructions):
        if isinstance(instructions,str): instructions=[instructions]
        enc=self.tokenizer(instructions,padding=True,truncation=True,max_length=77,return_tensors='pt').to(self.device)
        ctx=torch.no_grad() if self.frozen is True else torch.enable_grad()
        with ctx: out=self.model(**enc)
        return out.last_hidden_state[:,0,:]

class CLIPTextEncoder:
    def __init__(self,device='cuda',freeze=True):
        self.device=device; self.frozen=freeze
        self.tokenizer=CLIPTokenizer.from_pretrained('openai/clip-vit-base-patch32')
        self.model=CLIPTextModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
        self.embed_dim=512
        if freeze: [p.__setattr__('requires_grad',False) for p in self.model.parameters()]; self.model.eval()
        print(f"   ✅ CLIP Text (512-dim, frozen={freeze})")
    def encode(self,instructions):
        if isinstance(instructions,str): instructions=[instructions]
        enc=self.tokenizer(instructions,padding=True,truncation=True,max_length=77,return_tensors='pt').to(self.device)
        ctx=torch.no_grad() if self.frozen else torch.enable_grad()
        with ctx: out=self.model(**enc)
        return out.pooler_output

class BoWTextEncoder(nn.Module):
    def __init__(self,out_dim=256,vocab=8192,device='cuda',dropout=0.1):
        super().__init__()
        self.device=device; self.frozen=False; self.embed_dim=out_dim
        self._tok=DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
        self.embed=nn.Embedding(vocab,128,padding_idx=0)
        self.mlp=nn.Sequential(nn.Linear(128,out_dim),nn.GELU(),nn.Dropout(dropout),
                                nn.Linear(out_dim,out_dim),nn.LayerNorm(out_dim))
        nn.init.normal_(self.embed.weight,std=0.02); self.to(device)
        print(f"   ✅ BoW ({out_dim}-dim, fully trainable)")
    def encode(self,instructions):
        if isinstance(instructions,str): instructions=[instructions]
        enc=self._tok(instructions,padding=True,truncation=True,max_length=77,return_tensors='pt').to(self.device)
        ids=enc['input_ids'].clamp(max=8191); pad=(ids==0)
        emb=self.embed(ids); mask=(~pad).unsqueeze(-1).float()
        pooled=(emb*mask).sum(1)/mask.sum(1).clamp(min=1)
        return self.mlp(pooled)

print("✅ Block 6 done — MobileViT, PerceiverResampler, CausalTemporalAttention, text encoders defined")


✅ Block 6 done — MobileViT, PerceiverResampler, CausalTemporalAttention, text encoders defined


In [7]:
# ========================= BLOCK 7: MINI-VLA MODEL =========================
# ─── Build backbone ──────────────────────────────────────────────────────────
vit = MobileViTEncoder(drop_path_rate=MOBILEVIT_DROP_PATH)

class MiniVLAModel(nn.Module):
    """
    Mini-VLA: Vision-Language-Action model with action chunking.

    Architecture:
      video (B,C,F,H,W) ──► MobileViTEncoder ──► CausalTemporalAttention
                          ──► PerceiverResampler (per frame) ──► frame tokens
      text  (B,TEXT_DIM) ──► Linear projection ──► text token
      [text | frame_tokens] ──► Transformer ──► logits from text token
      logits ──► reshape ──► (B, CHUNK_SIZE, NUM_ACTIONS, ACTION_BINS)

    Action Chunking:
      Predicts CHUNK_SIZE future action steps simultaneously.
      At train time we supervise only position 0 (current timestep).
      At deployment, the robot executes all CHUNK_SIZE predictions, then re-queries
      — reducing inference frequency and improving temporal consistency.
    """
    def __init__(self, vit, text_dim=512, num_actions=7, action_bins=256,
                 depth=4, heads=8, dropout=0.2, num_latents=32, chunk_size=4):
        super().__init__()
        self.vit         = vit
        self.num_actions = num_actions
        self.action_bins = action_bins
        self.chunk_size  = chunk_size
        D = vit.embed_dim   # 128

        self.temporal     = CausalTemporalAttention(D, heads=4, dropout=dropout)
        self.token_learner= PerceiverResampler(D, num_latents=num_latents, depth=2,
                                               heads=4, dropout=dropout)
        self.text_proj    = nn.Sequential(
            nn.Linear(text_dim, D, bias=False), nn.LayerNorm(D), nn.Dropout(dropout))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=D, nhead=heads, dim_feedforward=D*4,
            dropout=dropout, batch_first=True, norm_first=True, activation='gelu')
        self.transformer = nn.TransformerEncoder(enc_layer, depth)

        # Action chunking head
        # Output: (B, chunk_size * num_actions * action_bins)
        self.to_logits = nn.Sequential(
            nn.LayerNorm(D),
            nn.Linear(D, D*2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(D*2, chunk_size * num_actions * action_bins))

    def forward(self, video, text_embeddings):
        B, C, F, H, W = video.shape
        flat    = rearrange(video, 'b c f h w -> (b f) c h w')
        vit_out = self.vit(flat)                                       # (B*F, D, h, w)
        vit_out = rearrange(vit_out, '(b f) d h w -> b d f h w', b=B, f=F)
        vit_out = self.temporal(vit_out)                               # (B, D, F, h, w)  causal
        vit_out = rearrange(vit_out, 'b d f h w -> (b f) d h w')
        tokens  = self.token_learner(vit_out)                          # (B*F, N, D)
        tokens  = rearrange(tokens, '(b f) n d -> b (f n) d', b=B, f=F)  # (B, F*N, D)
        text_tok= self.text_proj(text_embeddings).unsqueeze(1)         # (B, 1, D)
        attended= self.transformer(torch.cat([text_tok, tokens], dim=1))  # (B, 1+F*N, D)
        logits  = self.to_logits(attended[:, 0, :])                    # (B, chunk*A*BIN)
        return rearrange(logits, 'b (chunk a bin) -> b chunk a bin',
                         chunk=self.chunk_size, a=self.num_actions)    # (B, CHUNK, A, BIN)


model = MiniVLAModel(
    vit        = vit,
    text_dim   = TEXT_DIM,
    num_actions= NUM_ACTIONS,
    action_bins= ACTION_BINS,
    depth      = TRANSFORMER_DEPTH,
    heads      = TRANSFORMER_HEADS,
    dropout    = DROPOUT,
    num_latents= NUM_VISUAL_TOKENS,
    chunk_size = CHUNK_SIZE,
).to(DEVICE)

total_p = sum(p.numel() for p in model.parameters())
print(f"🤖 MiniVLA-MobileViT: {total_p:,} params  embed_dim={model.vit.embed_dim}")
with torch.no_grad():
    dv = torch.randn(2, 3, SEQ_LEN, TARGET_SIZE, TARGET_SIZE).to(DEVICE)
    dt = torch.randn(2, TEXT_DIM).to(DEVICE)
    out = model(dv, dt)
    print(f"   Sanity: {out.shape}")    # should be (2, CHUNK_SIZE, NUM_ACTIONS, ACTION_BINS)
    assert out.shape == (2, CHUNK_SIZE, NUM_ACTIONS, ACTION_BINS), f"Unexpected shape: {out.shape}"
    del dv, dt, out
torch.cuda.empty_cache(); print("✅ Block 7 done")


🤖 MiniVLA-MobileViT: 4,294,208 params  embed_dim=128
   Sanity: torch.Size([2, 4, 7, 256])
✅ Block 7 done


In [8]:
# ========================= BLOCK 8: TRAINING UTILITIES =========================
class ImprovedFocalLoss(nn.Module):
    def __init__(self,gamma=2.5,label_smoothing=0.1,temperature=1.0):
        super().__init__()
        self.gamma=gamma; self.label_smoothing=label_smoothing; self.temperature=temperature
    def forward(self,logits,targets):
        logits=logits/self.temperature
        ce=F.cross_entropy(logits,targets,reduction='none',label_smoothing=self.label_smoothing)
        p_t=F.softmax(logits,dim=-1).gather(1,targets.unsqueeze(1)).squeeze(1)
        return ((1-p_t)**self.gamma*ce).mean()

class ImprovedMixup(nn.Module):
    def __init__(self,alpha=0.3,use_cutmix=True,cutmix_prob=0.5):
        super().__init__()
        self.alpha=alpha; self.use_cutmix=use_cutmix; self.cutmix_prob=cutmix_prob
    def forward(self,x,y):
        lam=np.random.beta(self.alpha,self.alpha) if self.alpha>0 else 1.0
        idx=torch.randperm(x.size(0),device=x.device)
        if self.use_cutmix and np.random.rand()<self.cutmix_prob:
            W,H=x.size(-1),x.size(-2); cut=int(W*np.sqrt(1-lam))
            cx,cy=np.random.randint(W),np.random.randint(H)
            x1,y1=np.clip(cx-cut//2,0,W),np.clip(cy-cut//2,0,H)
            x2,y2=np.clip(cx+cut//2,0,W),np.clip(cy+cut//2,0,H)
            x[:,:,:,y1:y2,x1:x2]=x[idx,:,:,y1:y2,x1:x2]
            lam=1-(x2-x1)*(y2-y1)/(W*H)
        else: x=lam*x+(1-lam)*x[idx]
        return x,y,y[idx],lam

class Lion(torch.optim.Optimizer):
    def __init__(self,params,lr=1e-4,betas=(0.9,0.99),weight_decay=0.0):
        super().__init__(params,dict(lr=lr,betas=betas,weight_decay=weight_decay))
    @torch.no_grad()
    def step(self,closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                if group['weight_decay']>0: p.data.mul_(1-group['lr']*group['weight_decay'])
                state=self.state[p]
                if len(state)==0: state['exp_avg']=torch.zeros_like(p)
                exp_avg=state['exp_avg']; b1,b2=group['betas']
                update=exp_avg*b1+p.grad*(1-b1)
                p.add_(torch.sign(update),alpha=-group['lr'])
                exp_avg.mul_(b2).add_(p.grad,alpha=1-b2)

class ImprovedScheduler:
    def __init__(self,optimizer,warmup_steps=500,max_steps=10000,min_lr_ratio=0.1):
        self.optimizer=optimizer; self.warmup=warmup_steps; self.max_steps=max_steps
        self.min_ratio=min_lr_ratio; self.base_lrs=[g['lr'] for g in optimizer.param_groups]; self.step_count=0
    def step(self):
        self.step_count+=1
        for pg,blr in zip(self.optimizer.param_groups,self.base_lrs):
            if self.step_count<self.warmup: pg['lr']=blr*self.step_count/self.warmup
            else:
                prog=(self.step_count-self.warmup)/max(1,self.max_steps-self.warmup)
                pg['lr']=self.min_ratio*blr+(blr-self.min_ratio*blr)*0.5*(1+math.cos(math.pi*prog))
    def get_last_lr(self): return [g['lr'] for g in self.optimizer.param_groups]

class ModelEMA:
    def __init__(self,model,decay=0.9999):
        self.decay=decay; self.model=model
        self.shadow={n:p.data.clone() for n,p in model.named_parameters() if p.requires_grad}; self.backup={}
    def update(self,model):
        for n,p in model.named_parameters():
            if p.requires_grad: self.shadow[n]=self.decay*self.shadow[n]+(1-self.decay)*p.data
    def apply_shadow(self):
        for n,p in self.model.named_parameters():
            if p.requires_grad: self.backup[n]=p.data; p.data=self.shadow[n]
    def restore(self):
        for n,p in self.model.named_parameters():
            if p.requires_grad: p.data=self.backup[n]
        self.backup={}

class PerTaskMetrics:
    def __init__(self): self.task_losses=defaultdict(list)
    def reset(self): self.task_losses=defaultdict(list)
    def update(self,losses,task_ids):
        for l,t in zip(losses,task_ids): self.task_losses[t.item()].append(l.item())
    def get_average(self): return {t:np.mean(ls) for t,ls in self.task_losses.items()}

print("✅ Block 8 done")


✅ Block 8 done


In [9]:
# ========================= BLOCK 9: DATASET =========================
class CableRoutingDataset(Dataset):
    def __init__(self,split,mode="train",seq_len=SEQ_LEN,target_size=TARGET_SIZE,max_episodes=None):
        self.seq_len=seq_len; self.target_size=target_size; self.mode=mode; self.tokenizer=RT1Tokenizer()
        self.cache_dir=os.path.join(os.getcwd(),f"cable_cache_{mode}")
        self.img_path=os.path.join(self.cache_dir,"images.dat")
        self.act_path=os.path.join(self.cache_dir,"actions.npy")
        self.inst_path=os.path.join(self.cache_dir,"instructions.npy")
        self.meta_path=os.path.join(self.cache_dir,"metadata.npy")
        if os.path.exists(self.meta_path): self._load_cache()
        else: self._build_cache(split,max_episodes)
        self.indices=[]
        for ep_s,ep_e in self.episode_bounds:
            if ep_e-ep_s>=seq_len:
                for i in range(ep_s+seq_len,ep_e+1): self.indices.append(i)
        self._build_task_index(); self._setup_augmentation()
        size_gb=os.path.getsize(self.img_path)/1e9 if os.path.exists(self.img_path) else 0
        print(f"   ✅ {mode.upper()}: {len(self.episode_bounds)} ep, {len(self.indices):,} samples, "
              f"{self.num_tasks} tasks, {size_gb:.2f} GB")

    def _build_cache(self,split,max_episodes):
        os.makedirs(self.cache_dir,exist_ok=True)
        print(f"\n📂 Building cache: {self.mode} → {self.cache_dir}")
        ds=builder.as_dataset(split=split)
        total_frames,ep_count=0,0
        for ep in tqdm(ds,desc="   Counting"):
            if max_episodes and ep_count>=max_episodes: break
            total_frames+=sum(1 for _ in ep["steps"]); ep_count+=1
        print(f"   {ep_count} episodes, {total_frames} frames")
        ds2=builder.as_dataset(split=split)
        img_mmap=np.memmap(self.img_path,dtype=np.uint8,mode='w+',
                           shape=(total_frames,self.target_size,self.target_size,3))
        actions_list,instructions_list,episode_bounds=[],[],[]
        idx,ep_num=0,0
        for ep in tqdm(ds2,desc="   Writing",total=ep_count):
            if max_episodes and ep_num>=max_episodes: break
            ep_start=idx
            for step in ep["steps"]:
                img=step["observation"][IMG_KEY].numpy()
                if img.shape[0]!=self.target_size or img.shape[1]!=self.target_size:
                    img=np.array(PILImage.fromarray(img).resize((self.target_size,self.target_size),PILImage.BILINEAR))
                img_mmap[idx]=img
                act_vec=[np.atleast_1d(step["action"][k].numpy()).astype(np.float32) for k in ACT_KEYS]
                actions_list.append(np.concatenate(act_vec))
                inst=""
                if INST_KEY:
                    raw=step["observation"][INST_KEY].numpy()
                    inst=raw.decode() if isinstance(raw,bytes) else str(raw)
                instructions_list.append(inst or "route the cable")
                idx+=1
            episode_bounds.append((ep_start,idx)); ep_num+=1
        img_mmap.flush(); del img_mmap
        np.save(self.act_path,np.stack(actions_list))
        np.save(self.inst_path,np.array(instructions_list,dtype=object))
        np.save(self.meta_path,{'episode_bounds':episode_bounds,'n_frames':idx,'target_size':self.target_size})
        del actions_list,instructions_list; gc.collect(); self._load_cache()

    def _load_cache(self):
        meta=np.load(self.meta_path,allow_pickle=True).item()
        self.episode_bounds=meta['episode_bounds']; self.n_frames=meta['n_frames']
        self.all_images=np.memmap(self.img_path,dtype=np.uint8,mode='r',
                                  shape=(self.n_frames,self.target_size,self.target_size,3))
        self.all_actions=np.load(self.act_path)
        self.instructions=np.load(self.inst_path,allow_pickle=True).tolist()

    def _build_task_index(self):
        unique=sorted(set(self.instructions[i-1] for i in self.indices))
        self.instruction_to_id={inst:i for i,inst in enumerate(unique)}
        self.id_to_instruction={i:inst for inst,i in self.instruction_to_id.items()}
        self.idx_to_task={}; self.task_to_indices=defaultdict(list); self.task_counts=defaultdict(int)
        for idx in self.indices:
            inst=self.instructions[idx-1]; tid=self.instruction_to_id[inst]
            self.idx_to_task[idx]=tid; self.task_to_indices[tid].append(idx); self.task_counts[tid]+=1
        self.num_tasks=len(self.task_counts)

    def get_balanced_sampler(self):
        weights=np.array([1.0/self.task_counts[self.idx_to_task[i]] for i in self.indices])
        weights=weights/weights.sum()*len(weights)
        return WeightedRandomSampler(torch.DoubleTensor(weights),len(self.indices),replacement=True)

    def _setup_augmentation(self):
        if self.mode=="train":
            self.strong_aug=transforms.Compose([transforms.ToPILImage(),
                transforms.RandomApply([transforms.ColorJitter(0.4,0.4,0.4,0.2)],p=0.8),
                transforms.RandomResizedCrop(self.target_size,scale=(0.7,1.0)),
                transforms.RandomApply([transforms.GaussianBlur(5,(0.1,2.5))],p=0.5),
                transforms.ToTensor(),transforms.RandomErasing(p=0.3)])
            self.weak_aug=transforms.Compose([transforms.ToPILImage(),
                transforms.RandomResizedCrop(self.target_size,scale=(0.85,1.0)),transforms.ToTensor()])
        else:
            self.transform=transforms.Compose([transforms.ToPILImage(),
                transforms.Resize((self.target_size,self.target_size)),transforms.ToTensor()])

    def __len__(self): return len(self.indices)

    def __getitem__(self,idx):
        end=self.indices[idx]; start=end-self.seq_len
        instruction=self.instructions[end-1]; task_id=self.instruction_to_id[instruction]
        aug=self.strong_aug if (self.mode=="train" and self.task_counts[task_id]<500) else             (self.weak_aug if self.mode=="train" else self.transform)
        frames=[]
        for i in range(start,end):
            img=np.array(self.all_images[i])
            if img.dtype!=np.uint8:
                img=(img*255).astype(np.uint8) if img.max()<=1.0 else img.astype(np.uint8)
            frames.append(aug(img))
        video=torch.stack(frames); video=rearrange(video,'f c h w -> c f h w')
        action_tokens=self.tokenizer.encode(self.all_actions[end-1].astype(np.float32))
        return video,instruction,action_tokens,task_id

print("✅ Block 9 done")


✅ Block 9 done


In [10]:
# ========================= BLOCK 10: DATALOADERS =========================
print("\n" + "="*60)
print("INITIALIZING TEXT ENCODER + DATASETS")
print("="*60)

# Mini-VLA uses CLIP text encoder (512-dim, frozen)
text_encoder = CLIPTextEncoder(device=DEVICE, freeze=True)
assert text_encoder.embed_dim == TEXT_DIM, \
    f"TEXT_DIM mismatch: config={TEXT_DIM}, encoder={text_encoder.embed_dim}"

total_ep  = builder.info.splits["train"].num_examples
train_ep  = int(total_ep * SPLIT_RATIO)
print(f"\n📊 Split: {train_ep} train / {total_ep-train_ep} val")

dataset_train = CableRoutingDataset(split=f"train[:{train_ep}]",  mode="train")
dataset_val   = CableRoutingDataset(split=f"train[{train_ep}:]",  mode="val")
train_sampler = dataset_train.get_balanced_sampler()

loader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, sampler=train_sampler,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
    persistent_workers=NUM_WORKERS>0,
    prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS>0 else None)
loader_val   = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=NUM_WORKERS>0,
    prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS>0 else None)

v, inst, tok, tid = dataset_train[0]
print(f"\n🧪 Sanity: video={v.shape}, tokens={tok.shape}")
print(f"✅ Block 10 done — {len(loader_train)} train / {len(loader_val)} val batches")



INITIALIZING TEXT ENCODER + DATASETS
   ✅ CLIP Text (512-dim, frozen=True)

📊 Split: 1185 train / 297 val
   ✅ TRAIN: 1185 ep, 24,463 samples, 1 tasks, 4.56 GB
   ✅ VAL: 297 ep, 6,480 samples, 1 tasks, 1.19 GB

🧪 Sanity: video=torch.Size([3, 6, 224, 224]), tokens=torch.Size([7])
✅ Block 10 done — 24463 train / 6480 val batches


In [11]:
# ========================= BLOCK 11: TRAINING SETUP =========================
optimizer=Lion(model.parameters(),lr=LR,betas=LION_BETAS,weight_decay=WEIGHT_DECAY)
total_steps=len(loader_train)*EPOCHS//GRADIENT_ACCUM_STEPS
scheduler=ImprovedScheduler(optimizer,warmup_steps=WARMUP_STEPS,max_steps=total_steps)
criterion=ImprovedFocalLoss(gamma=FOCAL_GAMMA,label_smoothing=LABEL_SMOOTHING)
mixup=ImprovedMixup(alpha=MIXUP_ALPHA,use_cutmix=USE_CUTMIX)
ema=ModelEMA(model,decay=0.9999)
scaler=torch.amp.GradScaler('cuda',enabled=USE_BFLOAT16)
train_metrics=PerTaskMetrics(); val_metrics=PerTaskMetrics()
print(f"✅ Block 11 done — {total_steps} total steps")
print(f"   Lion LR={LR}, Focal γ={FOCAL_GAMMA}, EMA, Mixup+CutMix")


✅ Block 11 done — 15289 total steps
   Lion LR=3e-05, Focal γ=2.5, EMA, Mixup+CutMix


In [12]:
# ========================= BLOCK 12: TRAINING LOOP =========================
# NOTE: model(video, text) → (B, CHUNK_SIZE, NUM_ACTIONS, ACTION_BINS)
# We supervise only chunk position 0 (current timestep).
# Predicting CHUNK_SIZE ahead at each forward pass reduces inference frequency
# at deployment without changing the single-step dataset labelling.
best_val_loss=float('inf'); patience_counter=0
history={'train_loss':[],'val_loss':[],'lr':[],'throughput':[]}

print("\n" + "="*60)
print(f"🚀 TRAINING: {MODEL_NAME}")
print("="*60 + "\n")

for epoch in range(EPOCHS):
    model.train(); train_loss=0.0; train_metrics.reset()
    optimizer.zero_grad(); epoch_t0=time.time(); samples=0

    loop=tqdm(loader_train,desc=f"E{epoch+1}/{EPOCHS} [TRAIN]")
    for batch_idx,(video,instructions,target,task_ids) in enumerate(loop):
        video   =video.to(DEVICE,non_blocking=True)
        target  =target.to(DEVICE,non_blocking=True)
        task_ids=task_ids.to(DEVICE,non_blocking=True)

        with torch.no_grad(): text_emb=text_encoder.encode(list(instructions))

        video,target_a,target_b,lam=mixup(video,target)

        with torch.amp.autocast('cuda',dtype=AUTOCAST_DTYPE,enabled=USE_BFLOAT16):
            logits = model(video, text_emb)              # (B, CHUNK_SIZE, A, BIN)
            logits_c = logits[:, 0, :, :]               # (B, A, BIN) — supervise current step
            logits_flat = rearrange(logits_c, 'b a bin -> (b a) bin')
            ta_flat     = rearrange(target_a, 'b a -> (b a)')
            tb_flat     = rearrange(target_b, 'b a -> (b a)')
            loss = (lam*criterion(logits_flat,ta_flat)+(1-lam)*criterion(logits_flat,tb_flat)) / GRADIENT_ACCUM_STEPS

        scaler.scale(loss).backward()

        if (batch_idx+1)%GRADIENT_ACCUM_STEPS==0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
            scaler.step(optimizer); scaler.update()
            scheduler.step(); optimizer.zero_grad(); ema.update(model)

        train_loss+=loss.item()*GRADIENT_ACCUM_STEPS; samples+=video.size(0)
        loop.set_postfix(loss=f"{loss.item()*GRADIENT_ACCUM_STEPS:.3f}",
                         lr=f"{scheduler.get_last_lr()[0]:.2e}")

    # ── Validation ──────────────────────────────────────────────────────────
    model.eval(); val_loss=0.0; nb_val=0
    with torch.no_grad():
        for video,instructions,target,task_ids in loader_val:
            video =video.to(DEVICE); target=target.to(DEVICE)
            text_emb=text_encoder.encode(list(instructions))
            with torch.amp.autocast('cuda',dtype=AUTOCAST_DTYPE):
                logits = model(video, text_emb)          # (B, CHUNK_SIZE, A, BIN)
                logits_c = logits[:, 0, :, :]           # (B, A, BIN)
            val_loss+=criterion(
                rearrange(logits_c, 'b a bin -> (b a) bin'),
                rearrange(target,   'b a -> (b a)')).item()
            nb_val+=1

    tl=train_loss/len(loader_train); vl=val_loss/max(nb_val,1)
    elapsed=time.time()-epoch_t0; thr=samples/elapsed
    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['lr'].append(scheduler.get_last_lr()[0]); history['throughput'].append(thr)
    print(f"  E{epoch+1:3d}/{EPOCHS} | train={tl:.4f} val={vl:.4f} | "
          f"{thr:.1f} samp/s | {elapsed:.0f}s")

    if vl<best_val_loss:
        best_val_loss=vl; patience_counter=0
        torch.save({'model':model.state_dict(),'epoch':epoch+1,'val_loss':best_val_loss},
                   f'{MODEL_NAME}_best.pt')
        print(f"    💾 Saved (val={best_val_loss:.4f})")
    else:
        patience_counter+=1
        if patience_counter>=PATIENCE: print(f"  ⏹ Early stop E{epoch+1}"); break

print(f"\n✅ Training done. Best val loss: {best_val_loss:.4f}")



🚀 TRAINING: mini_vla_mobilevit



E1/80 [TRAIN]:   0%|          | 0/24463 [00:00<?, ?it/s]

  E  1/80 | train=5.2825 val=4.9634 | 9.3 samp/s | 2627s
    💾 Saved (val=4.9634)


E2/80 [TRAIN]:   0%|          | 0/24463 [00:00<?, ?it/s]

  E  2/80 | train=4.5451 val=3.8930 | 9.4 samp/s | 2605s
    💾 Saved (val=3.8930)


...

  E 24/80 | train=1.7951 val=1.9572 | 9.1 samp/s | 2683s


E25/80 [TRAIN]:   0%|          | 0/24463 [00:00<?, ?it/s]

  E 25/80 | train=1.7983 val=1.9720 | 9.3 samp/s | 2625s


E26/80 [TRAIN]:   0%|          | 0/24463 [00:00<?, ?it/s]

  E 26/80 | train=1.7913 val=1.9544 | 9.1 samp/s | 2674s
  ⏹ Early stop E26

✅ Training done. Best val loss: 1.8682


In [13]:
# ========================= BLOCK 13: TRAINING PLOTS =========================
fig,axes=plt.subplots(1,3,figsize=(18,5))
ne=len(history['train_loss']); ep_range=range(1,ne+1)
axes[0].plot(ep_range,history['train_loss'],label='Train',lw=2,marker='o',ms=3)
axes[0].plot(ep_range,history['val_loss'],label='Val',lw=2,marker='s',ms=3)
axes[0].set_title(f'Loss — {MODEL_NAME}',fontsize=14,fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(ep_range,history['lr'],lw=2,color='green',marker='o',ms=3)
axes[1].set_title('Learning Rate',fontsize=14,fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_yscale('log'); axes[1].grid(alpha=0.3)
axes[2].plot(ep_range,history['throughput'],lw=2,color='purple',marker='o',ms=3)
axes[2].axhline(np.mean(history['throughput']),color='red',ls='--',lw=2,
                label=f"Avg: {np.mean(history['throughput']):.0f}")
axes[2].set_title('Throughput (samp/s)',fontsize=14,fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].legend(); axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{MODEL_NAME}_training.png',dpi=150,bbox_inches='tight')
plt.show()
print("✅ Block 13 done")

✅ Block 13 done


In [14]:
# ========================= BLOCK 14: XAI DEPENDENCIES =========================
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','install','-q',
                       'lime','shap','grad-cam==1.5.5','imageio[ffmpeg]'])
import lime, lime.lime_image, shap, imageio
from lime.lime_image import LimeImageExplainer
from pytorch_grad_cam import GradCAM, HiResCAM, ShapleyCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from skimage.segmentation import mark_boundaries
print('✅ Block 14 done — XAI packages loaded')


✅ Block 14 done — XAI packages loaded



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [15]:
# ========================= BLOCK 15: XAI SETUP =========================
import cv2
from pathlib import Path
import matplotlib.backends.backend_agg as agg

# ─── Free TensorFlow GPU memory before XAI ────────────────────────────────
# After dataset loading is done, we don't need TF on GPU anymore
try:
    tf.config.set_visible_devices([], 'GPU')
    print('   TF GPU visibility cleared ✅')
except:
    pass
gc.collect()
torch.cuda.empty_cache()

# ── sklearn mock (required by LIME internals) ─────────────────────────────────
import types
from importlib.machinery import ModuleSpec

def _ensure_submod(full, **attrs):
    if full not in sys.modules:
        m=types.ModuleType(full); m.__spec__=ModuleSpec(name=full,loader=None)
        for k,v in attrs.items(): setattr(m,k,v)
        sys.modules[full]=m
        parent='.'.join(full.split('.')[:-1])
        if parent in sys.modules: setattr(sys.modules[parent],full.split('.')[-1],m)

class _LabelEncoder:
    def fit_transform(self,y): self.classes_=np.unique(y); return y
    def transform(self,y): return y

class _MinMaxScaler:
    def __init__(self,**kw): pass
    def fit_transform(self,X): return X
    def transform(self,X): return X

class _Ridge:
    def __init__(self,alpha=1.0,**kw): self.alpha=alpha
    def fit(self,X,y,sample_weight=None):
        W=np.diag(sample_weight) if sample_weight is not None else np.eye(len(y))
        A=X.T@W@X+self.alpha*np.eye(X.shape[1]); b=X.T@W@y
        self.coef_=np.linalg.solve(A,b); self.intercept_=0.0; return self
    def predict(self,X): return X@self.coef_+self.intercept_
    def score(self,X,y):
        p=self.predict(X); ss_res=((y-p)**2).sum(); ss_tot=((y-y.mean())**2).sum()
        return 1-ss_res/(ss_tot+1e-8)

def _check_random_state(seed):
    if seed is None or isinstance(seed,int): return np.random.RandomState(seed)
    return seed

_ensure_submod('sklearn')
_ensure_submod('sklearn.preprocessing',LabelEncoder=_LabelEncoder,MinMaxScaler=_MinMaxScaler)
_ensure_submod('sklearn.linear_model',Ridge=_Ridge)
_ensure_submod('sklearn.utils',check_random_state=_check_random_state)
_ensure_submod('sklearn.utils.validation',check_is_fitted=lambda *a,**k:None)
print('   sklearn mock ✅')

# ── Output tree ──────────��─────────────────────────────────────────────────────
XAI_ROOT = Path(f'xai_outputs_{MODEL_NAME}')
for sub in ['block_16_gradcam','block_17_lime','block_18_shap',
            'block_19_pdp_ice','block_20_fusions','block_21_dashboard']:
    (XAI_ROOT/sub).mkdir(parents=True,exist_ok=True)
print(f'✅ Output tree: {XAI_ROOT.resolve()}/')

# ── Load checkpoint ────────────────────────────────────────────────────────────
ckpt_path=f'{MODEL_NAME}_best.pt'
if os.path.exists(ckpt_path):
    raw=torch.load(ckpt_path,map_location=DEVICE,weights_only=False)
    state=raw.get('model',raw)
    missing,unexpected=model.load_state_dict(state,strict=False)
    if missing: print(f'   ⚠️  Missing ({len(missing)}): {missing[:3]}')
    print(f'✅ Loaded checkpoint (val={raw.get("val_loss","?")!r})')
else:
    print('⚠️  No checkpoint found ��� using current weights')
model.eval()

# ── Detect embed dim ──────────────────────────────────────────────────────────
with torch.no_grad():
    _p=torch.zeros(1,3,TARGET_SIZE,TARGET_SIZE,device=DEVICE)
    ACTUAL_EMBED_DIM=model.vit(_p).shape[1]; del _p
torch.cuda.empty_cache()
print(f'✅ Detected embed_dim = {ACTUAL_EMBED_DIM}')

# ── FrameWrapper for CAM ──────────────────────────────────────────────────────
class FrameWrapper(nn.Module):
    def __init__(self,base_model,embed_dim):
        super().__init__()
        self.vit=base_model.vit; self.pool=nn.AdaptiveAvgPool2d(1)
        self.head=nn.Linear(embed_dim,NUM_ACTIONS*ACTION_BINS)
    def forward(self,x): return self.head(self.pool(self.vit(x)).flatten(1))

frame_model=FrameWrapper(model,ACTUAL_EMBED_DIM).to(DEVICE).eval()
with torch.no_grad():
    _t=torch.zeros(1,3,TARGET_SIZE,TARGET_SIZE,device=DEVICE)
    assert frame_model(_t).shape==(1,NUM_ACTIONS*ACTION_BINS); del _t
torch.cuda.empty_cache()
print('✅ FrameWrapper sanity passed')

# ── Target layer (last conv block) ────────────────────────────────────────────
def _get_target_layer(fw):
    """Robustly find the last conv/bn layer in the ViT's final stage block."""
    stages = fw.vit.stages
    last_stage = stages[-1]

    # Get the last sub-block if the stage is iterable (Sequential / ModuleList)
    try:
        last_block = list(last_stage)[-1]
    except TypeError:
        last_block = last_stage  # stage is itself the block

    # Walk all named modules in reverse; return first Conv2d or BatchNorm2d found
    candidates = list(last_block.named_modules())
    for name, mod in reversed(candidates):
        if isinstance(mod, (nn.Conv2d, nn.BatchNorm2d)):
            print(f'   ✅ Target layer: {type(last_block).__name__}.{name}  →  {type(mod).__name__}')
            return [mod]

    # Last resort: return the block itself
    print(f'   ⚠️  Could not find Conv/BN — using {type(last_block).__name__} directly')
    return [last_block]

target_layer = _get_target_layer(frame_model)
cam_targets=[ClassifierOutputTarget(128)]

# ── Static XAI_N samples (keep on CPU, move to GPU only when needed) ──────────
XAI_N=4
xai_videos_cpu,xai_insts,xai_toks_cpu=[],[],[]
for v,inst,tok,tid in loader_val:
    xai_videos_cpu.append(v); xai_insts+=list(inst); xai_toks_cpu.append(tok)
    if len(xai_videos_cpu)>=XAI_N: break
xai_videos_cpu=torch.cat(xai_videos_cpu[:XAI_N])  # stays on CPU
xai_toks_cpu=torch.cat(xai_toks_cpu[:XAI_N])      # stays on CPU
xai_insts=xai_insts[:XAI_N]

# Only move to GPU when needed for inference
xai_videos=xai_videos_cpu.to(DEVICE)
with torch.no_grad():
    xai_text_embs=torch.stack([text_encoder.encode(ins) for ins in xai_insts]).squeeze(1)
print(f'✅ Static batch: video={xai_videos.shape}  text={xai_text_embs.shape}')

# ── 100-frame video sequence (store numpy on CPU) ─────────────────────────────
VID_FRAMES=100; VID_FPS=10
val_tf=transforms.Compose([transforms.ToPILImage(),
    transforms.Resize((TARGET_SIZE,TARGET_SIZE)),transforms.ToTensor()])
all_images_ds=dataset_val.all_images; all_insts_ds=np.array(dataset_val.instructions)
vid_start=SEQ_LEN; vid_end=min(vid_start+VID_FRAMES,len(all_images_ds))
VID_ACTUAL=vid_end-vid_start
VID_INSTRUCTION=str(all_insts_ds[vid_start])[:55]

vid_tensors=[]; vid_numpys=[]
for idx in range(vid_start,vid_end):
    img=all_images_ds[idx]
    if img.dtype!=np.uint8: img=(img*255).astype(np.uint8) if img.max()<=1.0 else img.astype(np.uint8)
    t=val_tf(img).unsqueeze(0)  # CPU tensor
    np_f=np.clip(t[0].numpy().transpose(1,2,0),0,1).astype(np.float32)
    vid_tensors.append(t.to(DEVICE))  # move to GPU for inference
    vid_numpys.append(np_f)
print(f'✅ Video sequence: {VID_ACTUAL} frames | "{VID_INSTRUCTION}"')

# ── Shared helpers ─────────────────────────────────────────────────────────────
rt1_tok=RT1Tokenizer()

@torch.no_grad()
def predict_logits(videos,text_embs):
    with torch.amp.autocast('cuda',dtype=AUTOCAST_DTYPE): return model(videos,text_embs).float()

def last_frame_np(video_tensor,idx=0,frame=-1):
    return video_tensor[idx,:,frame].cpu().float().permute(1,2,0).numpy().clip(0,1)

def normalise(arr):
    a=arr.astype(np.float32); return (a-a.min())/(a.max()-a.min()+1e-8)

def resize_to(arr_2d,h,w):
    return cv2.resize(arr_2d.astype(np.float32),(w,h),interpolation=cv2.INTER_LINEAR)

def fig_to_rgb(fig):
    """Convert matplotlib figure to RGB numpy array — compatible with all matplotlib versions."""
    fig.canvas.draw()
    # Modern matplotlib (3.8+) removed tostring_rgb; use buffer_rgba instead
    try:
        buf = fig.canvas.buffer_rgba()
        arr = np.asarray(buf)
        return arr[:, :, :3].copy()  # RGBA → RGB
    except AttributeError:
        # Fallback for older matplotlib
        buf=np.frombuffer(fig.canvas.tostring_rgb(),dtype=np.uint8)
        W,H=fig.canvas.get_width_height()
        return buf.reshape(H,W,3)

def make_header(canvas,text,h_bar=42):
    canvas[:h_bar,:]=22
    cv2.putText(canvas,text,(10,h_bar-12),cv2.FONT_HERSHEY_SIMPLEX,0.52,(220,220,220),1)

def save_video(frames_rgb,path,fps,gif=True):
    h,w=frames_rgb[0].shape[:2]; mp4=str(path)+'.mp4'
    wr=cv2.VideoWriter(mp4,cv2.VideoWriter_fourcc(*'mp4v'),fps,(w,h))
    for f in frames_rgb: wr.write(cv2.cvtColor(f,cv2.COLOR_RGB2BGR))
    wr.release()
    if gif:
        gif_p=str(path)+'.gif'; small=[cv2.resize(f,(w//2,h//2)) for f in frames_rgb]
        imageio.mimsave(gif_p,small,fps=fps,loop=0)
        print(f'      💾 {gif_p}  ({os.path.getsize(gif_p)/1024/1024:.1f} MB)')
    print(f'      💾 {mp4}')

def save_fig(fig,path):
    fig.savefig(str(path),dpi=150,bbox_inches='tight'); plt.show(); plt.close(fig)
    print(f'      💾 {path}')

def block_summary(folder):
    print(f'\n   📁 {folder.name}/')
    for f in sorted(folder.iterdir()):
        kb=f.stat().st_size/1024
        print(f'      📄 {f.name}  '+(f'{kb/1024:.1f} MB' if kb>1024 else f'{kb:.0f} KB'))

# ── Report VRAM ───────────────────────────────────────────────────────────────
torch.cuda.empty_cache()
if torch.cuda.is_available():
    free,total=torch.cuda.mem_get_info()
    print(f'\n   VRAM: {free/1024**3:.2f} GB free / {total/1024**3:.2f} GB total')

print('\n✅ Block 15 done — all helpers ready')
print(f'   Run order: 16 → 17 → 18 → 19 → 20 → 21')

   sklearn mock ✅
✅ Output tree: /media/sparsh/CaptainSlow/Programming Stuff/Programming Stuff/Code _n_ Stuff/All Projects/All Projects/VLAs/my work/xai_outputs_mini_vla_mobilevit/
✅ Loaded checkpoint (val=1.868163158275463)
✅ Detected embed_dim = 128
✅ FrameWrapper sanity passed
   ✅ Target layer: MobileViTBlock.fusion.1  →  BatchNorm2d
✅ Static batch: video=torch.Size([4, 3, 6, 224, 224])  text=torch.Size([4, 512])
✅ Video sequence: 100 frames | "route cable"

   VRAM: 5.13 GB free / 5.67 GB total

✅ Block 15 done — all helpers ready
   Run order: 16 → 17 → 18 → 19 → 20 → 21


In [16]:
# ========================= BLOCK 16: GRADCAM FAMILY =========================
# GradCAM | HiResCAM | ShapleyCAM
# Outputs: static, side-by-side video, filmstrip, evolution, trajectory

DIR16=XAI_ROOT/'block_16_gradcam'
CAM_METHODS={'GradCAM':GradCAM,'HiResCAM':HiResCAM,'ShapleyCAM':ShapleyCAM}
CAM_COLORS={'GradCAM':'royalblue','HiResCAM':'seagreen','ShapleyCAM':'crimson'}
FILMSTRIP_N=10

print('='*70); print('🔥 BLOCK 16: GradCAM Family'); print('='*70)

# ── 16-A  Static CAM ──────────────────────────────────────────────────────────
print('\n[1/6] Static CAM on XAI_N samples...')
cam_results={}
for name,CamCls in CAM_METHODS.items():
    overlays=[]
    with CamCls(model=frame_model,target_layers=target_layer) as cam:
        for i in range(XAI_N):
            frame_np=last_frame_np(xai_videos,i)
            frame_t=xai_videos[i,:,-1].unsqueeze(0)
            gray=cam(input_tensor=frame_t,targets=cam_targets)[0]
            ov=show_cam_on_image(frame_np,gray,use_rgb=True)
            overlays.append((frame_np,gray,ov))
    cam_results[name]=overlays; print(f'   ✅ {name}')

fig,axes=plt.subplots(XAI_N,len(CAM_METHODS)+1,figsize=(5*(len(CAM_METHODS)+1),5*XAI_N))
for i in range(XAI_N):
    axes[i,0].imshow(last_frame_np(xai_videos,i))
    axes[i,0].set_title(f'Original\n"{xai_insts[i][:28]}"',fontsize=8); axes[i,0].axis('off')
    for j,name in enumerate(CAM_METHODS):
        _,_,ov=cam_results[name][i]
        axes[i,j+1].imshow(ov); axes[i,j+1].set_title(name,fontsize=9,fontweight='bold')
        axes[i,j+1].axis('off')
plt.suptitle(f'GradCAM Family — {MODEL_NAME}',fontsize=13,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR16/'gradcam_static.png')

# ── 16-B  CAM on 100 frames ───────────────────────────────────────────────────
print(f'\n[2/6] Computing CAM on {VID_ACTUAL} frames...')
all_cam_data={}
for name,CamCls in CAM_METHODS.items():
    grays,overlays=[],[]
    print(f'   {name}:',end=' ',flush=True)
    with CamCls(model=frame_model,target_layers=target_layer) as cam:
        for f_idx in range(VID_ACTUAL):
            gray=cam(input_tensor=vid_tensors[f_idx],targets=cam_targets)[0]
            ov=show_cam_on_image(vid_numpys[f_idx],gray,use_rgb=True)
            grays.append(gray); overlays.append(ov)
            if (f_idx+1)%25==0: print(f_idx+1,end=' ',flush=True)
    all_cam_data[name]={'grays':grays,'overlays':overlays}; print('✅')
torch.cuda.empty_cache()

# ── 16-C  Side-by-side video ──────────────────────────────────────────────────
print(f'\n[3/6] Stitching side-by-side video...')
H,W=TARGET_SIZE,TARGET_SIZE; HBAR=42; n_panels=1+len(CAM_METHODS)
cw,ch=W*n_panels,H+HBAR; vid_frames=[]
for f_idx in range(VID_ACTUAL):
    canvas=np.zeros((ch,cw,3),dtype=np.uint8)
    make_header(canvas,f"GradCAM | {f_idx+1}/{VID_ACTUAL} | '{VID_INSTRUCTION}'")
    orig=(vid_numpys[f_idx]*255).astype(np.uint8)
    canvas[HBAR:,0:W]=orig
    cv2.putText(canvas,'Original',(4,HBAR+16),cv2.FONT_HERSHEY_SIMPLEX,0.44,(255,255,255),1)
    for m_idx,name in enumerate(CAM_METHODS):
        x0=(m_idx+1)*W; canvas[HBAR:,x0:x0+W]=all_cam_data[name]['overlays'][f_idx]
        cv2.putText(canvas,name,(x0+4,HBAR+16),cv2.FONT_HERSHEY_SIMPLEX,0.44,(255,255,255),1)
    vid_frames.append(canvas)
save_video(vid_frames,DIR16/'gradcam_sidebyside',VID_FPS)

# ── 16-D  Filmstrip ───────────────────────────────────────────────────────────
print('\n[4/6] Filmstrip...')
strip_idxs=list(range(0,VID_ACTUAL,FILMSTRIP_N)); n_strip=len(strip_idxs)
fig,axes=plt.subplots(1+len(CAM_METHODS),n_strip,figsize=(3*n_strip,3*(1+len(CAM_METHODS))))
if n_strip==1: axes=axes[:,np.newaxis]
for col,f_idx in enumerate(strip_idxs):
    axes[0,col].imshow(vid_numpys[f_idx]); axes[0,col].set_title(f'F{f_idx+1}',fontsize=7)
    axes[0,col].axis('off')
    if col==0: axes[0,col].set_ylabel('Original',fontsize=9,fontweight='bold')
    for row,name in enumerate(CAM_METHODS,start=1):
        axes[row,col].imshow(all_cam_data[name]['overlays'][f_idx]); axes[row,col].axis('off')
        if col==0: axes[row,col].set_ylabel(name,fontsize=9,fontweight='bold')
plt.suptitle(f'GradCAM Filmstrip (every {FILMSTRIP_N} frames)\n"{VID_INSTRUCTION}"',fontsize=12,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR16/'gradcam_filmstrip.png')

# ── 16-E  Attention evolution ─────────────────────────────────────────────────
print('\n[5/6] Attention evolution...')
attn_stats={m:{'cx':[],'cy':[],'spread':[],'peak':[]} for m in CAM_METHODS}
for name in CAM_METHODS:
    for f_idx in range(VID_ACTUAL):
        gray=all_cam_data[name]['grays'][f_idx]; hc,wc=gray.shape; total=gray.sum()+1e-8
        yy,xx=np.mgrid[0:hc,0:wc]
        cx=(xx*gray).sum()/total; cy=(yy*gray).sum()/total
        spread=np.sqrt((((xx-cx)**2+(yy-cy)**2)*gray).sum()/total)
        attn_stats[name]['cx'].append(cx/wc); attn_stats[name]['cy'].append(cy/hc)
        attn_stats[name]['spread'].append(spread/max(hc,wc)); attn_stats[name]['peak'].append(gray.max())
frames_ax=np.arange(1,VID_ACTUAL+1)
fig,axes=plt.subplots(2,2,figsize=(16,9))
metrics=[('cx','Attention Center X','Norm. X'),('cy','Attention Center Y','Norm. Y'),
         ('spread','Attention Spread σ','Norm. spread'),('peak','Peak Activation','Max CAM')]
for ax,(key,title,ylabel) in zip(axes.flat,metrics):
    for name in CAM_METHODS:
        ax.plot(frames_ax,attn_stats[name][key],color=CAM_COLORS[name],label=name,lw=1.6,alpha=0.85)
    ax.set_title(title,fontweight='bold'); ax.set_xlabel('Frame'); ax.set_ylabel(ylabel)
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle(f'Temporal Attention Evolution — "{VID_INSTRUCTION}"',fontsize=13,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR16/'gradcam_evolution.png')

# ── 16-F  Trajectory map ──────────────────────────────────────────────────────
print('\n[6/6] Attention trajectory...')
fig,axes=plt.subplots(1,len(CAM_METHODS),figsize=(6*len(CAM_METHODS),6))
mid_frame=vid_numpys[VID_ACTUAL//2]; time_cols=plt.cm.coolwarm(np.linspace(0,1,VID_ACTUAL))
for ax,name in zip(axes,CAM_METHODS):
    cx=np.array(attn_stats[name]['cx'])*TARGET_SIZE; cy=np.array(attn_stats[name]['cy'])*TARGET_SIZE
    sp=np.array(attn_stats[name]['spread'])*TARGET_SIZE
    ax.imshow(mid_frame); ax.plot(cx,cy,color='white',lw=1.2,alpha=0.35,zorder=1)
    for f_idx in range(VID_ACTUAL):
        ax.scatter(cx[f_idx],cy[f_idx],c=[time_cols[f_idx]],s=max(8,sp[f_idx]*3),
                   alpha=0.7,edgecolors='white',linewidths=0.25,zorder=2)
    ax.scatter(cx[0],cy[0],c='cyan',s=140,marker='^',edgecolors='white',lw=2,zorder=3,label='Start')
    ax.scatter(cx[-1],cy[-1],c='magenta',s=140,marker='v',edgecolors='white',lw=2,zorder=3,label='End')
    ax.set_title(f'{name} — Trajectory',fontsize=12,fontweight='bold')
    ax.legend(fontsize=9); ax.axis('off')
plt.suptitle(f'Attention Path  blue=early → red=late  size∝spread\n"{VID_INSTRUCTION}"',fontsize=12,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR16/'gradcam_trajectory.png')

block_summary(DIR16); print('\n✅ Block 16 done')


🔥 BLOCK 16: GradCAM Family

[1/6] Static CAM on XAI_N samples...
   ✅ GradCAM
   ✅ HiResCAM
   ✅ ShapleyCAM
      💾 xai_outputs_mini_vla_mobilevit/block_16_gradcam/gradcam_static.png

[2/6] Computing CAM on 100 frames...
   GradCAM: 25 50 75 100 ✅
   HiResCAM: 25 50 75 100 ✅
   ShapleyCAM: 25 50 75 100 ✅

[3/6] Stitching side-by-side video...
      💾 xai_outputs_mini_vla_mobilevit/block_16_gradcam/gradcam_sidebyside.gif  (3.4 MB)
      💾 xai_outputs_mini_vla_mobilevit/block_16_gradcam/gradcam_sidebyside.mp4

[4/6] Filmstrip...
      💾 xai_outputs_mini_vla_mobilevit/block_16_gradcam/gradcam_filmstrip.png

[5/6] Attention evolution...
      💾 xai_outputs_mini_vla_mobilevit/block_16_gradcam/gradcam_evolution.png

[6/6] Attention trajectory...
      💾 xai_outputs_mini_vla_mobilevit/block_16_gradcam/gradcam_trajectory.png

   📁 block_16_gradcam/
      📄 gradcam_evolution.png  385 KB
      📄 gradcam_filmstrip.png  6.9 MB
      📄 gradcam_sidebyside.gif  3.4 MB
      📄 gradcam_sidebyside.mp4  

In [17]:
# ========================= BLOCK 17: LIME =========================
DIR17=XAI_ROOT/'block_17_lime'
LIME_SAMPLES_STATIC=128; LIME_SAMPLES_VIDEO=50; LIME_TOP_K=5; FILMSTRIP_N=10

print('='*70); print('🍋 BLOCK 17: LIME'); print('='*70)

def lime_predict(images_np):
    tensors=[torch.from_numpy(img.astype(np.float32)).permute(2,0,1) for img in images_np]
    batch=torch.stack(tensors).to(DEVICE)
    video=batch.unsqueeze(2).expand(-1,-1,SEQ_LEN,-1,-1)
    text_emb=xai_text_embs[0:1].expand(len(images_np),-1)
    with torch.no_grad(), torch.amp.autocast('cuda',dtype=AUTOCAST_DTYPE):
        logits=model(video,text_emb).float()              # (N, CHUNK, A, BIN)
    bins=torch.arange(ACTION_BINS,device=DEVICE).float()
    probs=torch.softmax(logits,dim=-1)                    # (N, CHUNK, A, BIN)
    actions=((probs*bins[None,None,None,:]).sum(-1)/float(ACTION_BINS-1))  # (N, CHUNK, A)
    return actions[:,0,:].cpu().numpy()                   # chunk 0 → (N, NUM_ACTIONS)

def lime_overlay(frame_np,mask):
    mask_2d=mask[:,:,0] if mask.ndim==3 else mask; mask_2d=(mask_2d==1)
    ov=frame_np.copy(); green=np.array([0.,0.85,0.2],dtype=np.float32)
    ov[mask_2d]=ov[mask_2d]*0.35+green*0.65
    return np.clip(ov,0,1)

explainer_lime=LimeImageExplainer()

# ── 17-A  Static LIME ─────────────────────────────────────────────────────────
print(f'\n[1/5] Static LIME ({LIME_SAMPLES_STATIC} samples × {XAI_N} frames)...')
lime_results=[]
for i in range(XAI_N):
    frame_np=last_frame_np(xai_videos,i)
    exp=explainer_lime.explain_instance(frame_np.astype(np.float64),lime_predict,
        top_labels=NUM_ACTIONS,hide_color=0,num_samples=LIME_SAMPLES_STATIC,num_features=20)
    _,mask=exp.get_image_and_mask(label=0,positive_only=True,num_features=LIME_TOP_K,hide_rest=False)
    mask_2d=mask[:,:,0] if mask.ndim==3 else mask
    lime_results.append((frame_np,mask_2d,exp))
    print(f'   sample {i+1}/{XAI_N} done')

fig,axes=plt.subplots(2,XAI_N,figsize=(5*XAI_N,10))
for i,(frame_np,mask_2d,_) in enumerate(lime_results):
    axes[0,i].imshow(frame_np); axes[0,i].set_title(f'Sample {i+1}\n"{xai_insts[i][:25]}"',fontsize=8)
    axes[0,i].axis('off')
    axes[1,i].imshow(lime_overlay(frame_np,mask_2d))
    axes[1,i].set_title(f'Top-{LIME_TOP_K} superpixels',fontsize=8); axes[1,i].axis('off')
plt.suptitle(f'LIME — {MODEL_NAME}',fontsize=13,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR17/'lime_static.png')

# ── 17-B  LIME video ──────────────────────────────────────────────────────────
print(f'\n[2/5] LIME video ({LIME_SAMPLES_VIDEO} samples × {VID_ACTUAL} frames)...')
lime_masks=[]; lime_overlays=[]; active_area=[]; t0=time.time()
for f_idx in range(VID_ACTUAL):
    frame_np=vid_numpys[f_idx]
    exp=explainer_lime.explain_instance(frame_np.astype(np.float64),lime_predict,
        top_labels=NUM_ACTIONS,hide_color=0,num_samples=LIME_SAMPLES_VIDEO,num_features=20)
    _,mask=exp.get_image_and_mask(label=0,positive_only=True,num_features=LIME_TOP_K,hide_rest=False)
    mask_2d=mask[:,:,0] if mask.ndim==3 else mask
    ov_float=lime_overlay(frame_np,mask_2d)
    lime_masks.append(mask_2d); lime_overlays.append((ov_float*255).astype(np.uint8))
    active_area.append(mask_2d.mean())
    if (f_idx+1)%10==0:
        elapsed=time.time()-t0; eta=elapsed/(f_idx+1)*(VID_ACTUAL-f_idx-1)
        print(f'   [{f_idx+1:3d}/{VID_ACTUAL}]  active={mask_2d.mean():.3f}  elapsed={elapsed:.0f}s  ETA={eta:.0f}s')
print(f'   ✅ LIME done in {time.time()-t0:.0f}s')

# ── 17-C  Stitch LIME video ───────────────────────────────────────────────────
print('\n[3/5] Stitching LIME video...')
H,W=TARGET_SIZE,TARGET_SIZE; HBAR=42; cw,ch=W*2,H+HBAR; frames_rgb=[]
for f_idx in range(VID_ACTUAL):
    canvas=np.zeros((ch,cw,3),dtype=np.uint8)
    make_header(canvas,f"LIME | {f_idx+1}/{VID_ACTUAL} | active={active_area[f_idx]:.1%} | '{VID_INSTRUCTION}'")
    canvas[HBAR:,0:W]=(vid_numpys[f_idx]*255).astype(np.uint8)
    cv2.putText(canvas,'Original',(4,HBAR+16),cv2.FONT_HERSHEY_SIMPLEX,0.44,(255,255,255),1)
    canvas[HBAR:,W:2*W]=lime_overlays[f_idx]
    cv2.putText(canvas,f'LIME top-{LIME_TOP_K}',(W+4,HBAR+16),cv2.FONT_HERSHEY_SIMPLEX,0.44,(255,255,255),1)
    frames_rgb.append(canvas)
save_video(frames_rgb,DIR17/'lime_video',VID_FPS)

# ── 17-D  Filmstrip ───────────────────────────────────────────────────────────
print('\n[4/5] Filmstrip...')
strip_idxs=list(range(0,VID_ACTUAL,FILMSTRIP_N)); n_strip=len(strip_idxs)
fig,axes=plt.subplots(2,n_strip,figsize=(3*n_strip,6))
if n_strip==1: axes=axes[:,np.newaxis]
for col,f_idx in enumerate(strip_idxs):
    axes[0,col].imshow(vid_numpys[f_idx]); axes[0,col].set_title(f'F{f_idx+1}',fontsize=7)
    axes[0,col].axis('off')
    if col==0: axes[0,col].set_ylabel('Original',fontsize=9,fontweight='bold')
    axes[1,col].imshow(lime_overlays[f_idx]); axes[1,col].axis('off')
    if col==0: axes[1,col].set_ylabel('LIME',fontsize=9,fontweight='bold')
plt.suptitle(f'LIME Filmstrip (every {FILMSTRIP_N} frames)\n"{VID_INSTRUCTION}"',fontsize=12,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR17/'lime_filmstrip.png')

# ── 17-E  Activity analysis ───────────────────────────────────────────────────
print('\n[5/5] Activity analysis...')
active_arr=np.array(active_area); mask_stack=np.stack(lime_masks); mean_mask=mask_stack.mean(axis=0)
def rolling(arr,w=10): return np.convolve(arr,np.ones(w)/w,mode='same')
fig,axes=plt.subplots(1,3,figsize=(18,5))
frames_ax=np.arange(1,VID_ACTUAL+1)
axes[0].fill_between(frames_ax,active_arr,alpha=0.25,color='limegreen')
axes[0].plot(frames_ax,active_arr,color='green',lw=1.2,alpha=0.55,label='Per-frame')
axes[0].plot(frames_ax,rolling(active_arr),color='darkgreen',lw=2.2,label='Rolling mean')
axes[0].set_title('Highlighted Region Size',fontweight='bold'); axes[0].set_xlabel('Frame')
axes[0].set_ylabel('Fraction highlighted'); axes[0].legend(); axes[0].grid(alpha=0.3)
mid=vid_numpys[VID_ACTUAL//2]
axes[1].imshow(mid); axes[1].imshow(mean_mask,cmap='Greens',alpha=0.6)
axes[1].set_title('Aggregate LIME Attention\n(mean mask)',fontweight='bold'); axes[1].axis('off')
axes[2].imshow(mean_mask,cmap='hot'); axes[2].set_title('Mean Superpixel Heatmap',fontweight='bold')
axes[2].axis('off')
plt.suptitle(f'LIME Temporal Analysis — "{VID_INSTRUCTION}"',fontsize=13,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR17/'lime_activity.png')

block_summary(DIR17); print('\n✅ Block 17 done')

🍋 BLOCK 17: LIME

[1/5] Static LIME (128 samples × 4 frames)...


  0%|          | 0/128 [00:00<?, ?it/s]

   sample 1/4 done


  0%|          | 0/128 [00:00<?, ?it/s]

   sample 2/4 done


...

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

   [100/100]  active=0.129  elapsed=136s  ETA=0s
   ✅ LIME done in 136s

[3/5] Stitching LIME video...
      💾 xai_outputs_mini_vla_mobilevit/block_17_lime/lime_video.gif  (2.0 MB)
      💾 xai_outputs_mini_vla_mobilevit/block_17_lime/lime_video.mp4

[4/5] Filmstrip...
      💾 xai_outputs_mini_vla_mobilevit/block_17_lime/lime_filmstrip.png

[5/5] Activity analysis...
      💾 xai_outputs_mini_vla_mobilevit/block_17_lime/lime_activity.png

   📁 block_17_lime/
      📄 lime_activity.png  684 KB
      📄 lime_filmstrip.png  3.3 MB
      📄 lime_static.png  886 KB
      📄 lime_video.gif  2.0 MB
      📄 lime_video.mp4  1.3 MB

✅ Block 17 done


In [18]:
# ========================= BLOCK 18: SHAP =========================
DIR18=XAI_ROOT/'block_18_shap'
print('='*70); print('🔵 BLOCK 18: SHAP (GradientExplainer → batched GPU occlusion)'); print('='*70)

class SHAPFrameWrapper(nn.Module):
    def __init__(self,base,embed_dim):
        super().__init__()
        self.vit=base.vit; self.pool=nn.AdaptiveAvgPool2d(1); self.head=nn.Linear(embed_dim,256)
    def forward(self,x): return self.head(self.pool(self.vit(x)).flatten(1))

torch.cuda.empty_cache(); gc.collect()
shap_model=SHAPFrameWrapper(model,ACTUAL_EMBED_DIM).to(DEVICE).eval()
with torch.no_grad():
    _t=torch.zeros(1,3,TARGET_SIZE,TARGET_SIZE,device=DEVICE)
    assert shap_model(_t).shape==(1,256); del _t
torch.cuda.empty_cache(); print('   SHAPFrameWrapper ✅')

# ── helper: resize shap map to match frame if shapes differ ──────────────────
def resize_to_frame(shap_map, frame_np):
    h, w = frame_np.shape[:2]
    if shap_map.shape != (h, w):
        shap_map = cv2.resize(shap_map.astype(np.float32),(w,h),interpolation=cv2.INTER_LINEAR)
    return shap_map

# ── helper: batched GPU occlusion (no gradients → minimal VRAM) ──────────────
def gpu_occlusion(tensor_1chw, patch=28, stride=14, batch_size=16):
    """
    Batch multiple occluded images into a single forward pass.
    No gradients → uses ~batch_size × single-image VRAM only.
    patch=28, stride=14 → (15×15)=225 patches → 15 batches of 16.
    """
    _,C,H,W = tensor_1chw.shape
    with torch.no_grad():
        baseline = shap_model(tensor_1chw).sum().item()

    # Pre-compute all patch positions
    ys = list(range(0, H-patch+1, stride))
    xs = list(range(0, W-patch+1, stride))
    positions = [(y,x) for y in ys for x in xs]

    heat   = np.zeros((H,W), dtype=np.float32)
    counts = np.zeros((H,W), dtype=np.float32)

    # Process in batches
    for b_start in range(0, len(positions), batch_size):
        batch_pos = positions[b_start:b_start+batch_size]
        batch = tensor_1chw.expand(len(batch_pos),-1,-1,-1).clone()
        for k,(y,x) in enumerate(batch_pos):
            batch[k,:,y:y+patch,x:x+patch] = 0.5
        with torch.no_grad():
            vals = shap_model(batch).sum(dim=1).cpu().numpy()  # (B,)
        for k,(y,x) in enumerate(batch_pos):
            drop = baseline - vals[k]
            heat[y:y+patch,x:x+patch]   += drop
            counts[y:y+patch,x:x+patch] += 1
        del batch

    counts = np.maximum(counts,1)
    return heat / counts

# ── helper: compute one attribution map — SHAP first, occlusion on OOM ───────
def compute_shap_map(single, bg, label=''):
    # Attempt 1: GPU GradientExplainer (low nsamples to save VRAM)
    try:
        torch.cuda.empty_cache(); gc.collect()
        exp = shap.GradientExplainer(shap_model, bg)
        sv  = exp.shap_values(single, nsamples=20)
        sv_map = np.abs(np.array(sv)).mean(axis=0)[0].mean(axis=0)
        del exp; torch.cuda.empty_cache()
        return sv_map
    except (RuntimeError, torch.cuda.OutOfMemoryError):
        print(f'   ⚠️  [{label}] SHAP OOM → batched GPU occlusion')
        torch.cuda.empty_cache(); gc.collect()

    # Attempt 2: batched GPU occlusion — no backprop, stays on GPU
    sv_map = gpu_occlusion(single, patch=28, stride=14, batch_size=16)
    torch.cuda.empty_cache(); gc.collect()
    return sv_map

# ── 18-A  Static SHAP ─────────────────────────────────────────────────────────
print(f'\n[1/5] Static SHAP ({XAI_N} samples)...')
shap_agg_vis=[]
for i in range(XAI_N):
    single = xai_videos[i:i+1,:,-1].clone()
    bg_one = xai_videos[(i+1)%XAI_N:((i+1)%XAI_N)+1,:,-1].clone()
    sv_map = compute_shap_map(single, bg_one, label=f'static-{i+1}')
    frame_np = last_frame_np(xai_videos,i)
    sv_map = resize_to_frame(sv_map, frame_np)
    shap_agg_vis.append(sv_map)
    del single, bg_one
    torch.cuda.empty_cache(); gc.collect()
    print(f'   sample {i+1}/{XAI_N} done')
shap_agg_vis=np.stack(shap_agg_vis)

fig,axes=plt.subplots(2,XAI_N,figsize=(5*XAI_N,10))
for i in range(XAI_N):
    frame_np=last_frame_np(xai_videos,i)
    axes[0,i].imshow(frame_np); axes[0,i].set_title(f'Input Frame {i+1}',fontsize=9); axes[0,i].axis('off')
    smap=shap_agg_vis[i]; smap=(smap-smap.min())/(smap.max()-smap.min()+1e-8)
    im=axes[1,i].imshow(smap,cmap='hot',vmin=0,vmax=1)
    axes[1,i].set_title('|SHAP| Attribution',fontsize=8); axes[1,i].axis('off')
    plt.colorbar(im,ax=axes[1,i],fraction=0.046)
plt.suptitle(f'SHAP / Occlusion — {MODEL_NAME}',fontsize=13,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR18/'shap_static.png')

# ── 18-B  Attribution video ───────────────────────────────────────────────────
print(f"\n[2/5] Attribution video ({VID_ACTUAL} frames)...")
cmap_hot=plt.cm.hot; shap_vid_maps=[]; t0=time.time()
SHAP_CHUNK=5
for f_idx in range(VID_ACTUAL):
    single = vid_tensors[f_idx].clone()
    bg     = vid_tensors[(f_idx+1)%VID_ACTUAL].clone()
    sv_map = compute_shap_map(single, bg, label=f'vid-{f_idx+1}')
    sv_map = resize_to_frame(sv_map, vid_numpys[f_idx])
    shap_vid_maps.append(sv_map.copy())
    del single, bg, sv_map
    if (f_idx+1) % SHAP_CHUNK == 0:
        torch.cuda.empty_cache(); gc.collect()
        elapsed=time.time()-t0; eta=elapsed/(f_idx+1)*(VID_ACTUAL-f_idx-1)
        print(f'   [{f_idx+1:3d}/{VID_ACTUAL}]  elapsed={elapsed:.0f}s  ETA={eta:.0f}s')
torch.cuda.empty_cache(); gc.collect()
print(f'   ✅ done in {time.time()-t0:.0f}s')
all_shap_arr=np.stack(shap_vid_maps); global_max=all_shap_arr.max()+1e-8
shap_norm_maps=all_shap_arr/global_max

def shap_blend(frame_np, shap_norm):
    shap_norm=resize_to_frame(shap_norm, frame_np)
    heat=cmap_hot(shap_norm)[:,:,:3].astype(np.float32)
    return np.clip(frame_np*0.45+heat*0.55, 0, 1)

# ── 18-C  Stitch video ────────────────────────────────────────────────────────
print('\n[3/5] Stitching video...')
H,W=TARGET_SIZE,TARGET_SIZE; HBAR=42; cw,ch=W*3,H+HBAR; frames_rgb=[]
for f_idx in range(VID_ACTUAL):
    frame_np=vid_numpys[f_idx]
    shap_norm=resize_to_frame(shap_norm_maps[f_idx], frame_np)
    heat=cmap_hot(shap_norm)[:,:,:3]; blend=shap_blend(frame_np,shap_norm)
    canvas=np.zeros((ch,cw,3),dtype=np.uint8)
    make_header(canvas,f"SHAP | {f_idx+1}/{VID_ACTUAL} | '{VID_INSTRUCTION}'")
    canvas[HBAR:,0:W]=(frame_np*255).astype(np.uint8)
    canvas[HBAR:,W:2*W]=(heat*255).astype(np.uint8)
    canvas[HBAR:,2*W:3*W]=(blend*255).astype(np.uint8)
    for lbl,x0 in [('Original',0),('Attribution heat',W),('Overlay',2*W)]:
        cv2.putText(canvas,lbl,(x0+4,HBAR+16),cv2.FONT_HERSHEY_SIMPLEX,0.44,(255,255,255),1)
    frames_rgb.append(canvas)
save_video(frames_rgb,DIR18/'shap_video',VID_FPS)

# ── 18-D  Filmstrip ───────────────────────────────────────────────────────────
print('\n[4/5] Filmstrip...')
strip_idxs=list(range(0,VID_ACTUAL,10)); n_strip=len(strip_idxs)
fig,axes=plt.subplots(3,n_strip,figsize=(3*n_strip,9))
if n_strip==1: axes=axes[:,np.newaxis]
for col,f_idx in enumerate(strip_idxs):
    frame_np=vid_numpys[f_idx]
    shap_norm=resize_to_frame(shap_norm_maps[f_idx], frame_np)
    blend=shap_blend(frame_np,shap_norm)
    for row,(img,lbl,kw) in enumerate([(frame_np,'Original',{}),
                                        (shap_norm,'Attribution',{'cmap':'hot','vmin':0,'vmax':1}),
                                        (blend,'Overlay',{})]):
        axes[row,col].imshow(img,**kw); axes[row,col].set_title(f'F{f_idx+1}',fontsize=7)
        axes[row,col].axis('off')
        if col==0: axes[row,col].set_ylabel(lbl,fontsize=9,fontweight='bold')
plt.suptitle(f'Attribution Filmstrip — "{VID_INSTRUCTION}"',fontsize=12,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR18/'shap_filmstrip.png')

# ── 18-E  Mean attribution ────────────────────────────────────────────────────
print('\n[5/5] Mean attribution...')
mean_shap=shap_norm_maps.mean(axis=0); mid=vid_numpys[VID_ACTUAL//2]
mean_shap=resize_to_frame(mean_shap, mid)
fig,axes=plt.subplots(1,3,figsize=(15,5))
axes[0].imshow(mid); axes[0].set_title('Middle Frame',fontweight='bold'); axes[0].axis('off')
axes[1].imshow(mean_shap,cmap='hot',vmin=0,vmax=1)
axes[1].set_title('Mean Attribution — 100 frames',fontweight='bold'); axes[1].axis('off')
axes[2].imshow(shap_blend(mid,mean_shap))
axes[2].set_title('Mean Overlay',fontweight='bold'); axes[2].axis('off')
plt.suptitle(f'Aggregate Attribution — "{VID_INSTRUCTION}"',fontsize=13,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR18/'shap_mean.png')

block_summary(DIR18); print('\n✅ Block 18 done')

🔵 BLOCK 18: SHAP (GradientExplainer → batched GPU occlusion)
   SHAPFrameWrapper ✅

[1/5] Static SHAP (4 samples)...
   ⚠️  [static-1] SHAP OOM → batched GPU occlusion
   sample 1/4 done
   ⚠️  [static-2] SHAP OOM → batched GPU occlusion
   sample 2/4 done
   ⚠️  [static-3] SHAP OOM → batched GPU occlusion
   sample 3/4 done
   ⚠️  [static-4] SHAP OOM → batched GPU occlusion
   sample 4/4 done
      💾 xai_outputs_mini_vla_mobilevit/block_18_shap/shap_static.png

[2/5] Attribution video (100 frames)...
   ⚠️  [vid-1] SHAP OOM → batched GPU occlusion
   ⚠️  [vid-2] SHAP OOM → batched GPU occlusion
   ⚠️  [vid-3] SHAP OOM → batched GPU occlusion
   ⚠️  [vid-4] SHAP OOM → batched GPU occlusion
   ⚠️  [vid-5] SHAP OOM → batched GPU occlusion
   [  5/100]  elapsed=10s  ETA=199s
   ⚠️  [vid-6] SHAP OOM → batched GPU occlusion
   ⚠️  [vid-7] SHAP OOM → batched GPU occlusion
   ⚠️  [vid-8] SHAP OOM → batched GPU occlusion
   ⚠️  [vid-9] SHAP OOM → batched GPU occlusion
   ⚠️  [vid-10] SHAP OOM 

In [19]:
# ========================= BLOCK 19: PDP + ICE + VALIDATION =========================
DIR19=XAI_ROOT/'block_19_pdp_ice'
ACTION_NAMES=['wx','wy','wz','rx','ry','rz','term'][:NUM_ACTIONS]
ACTION_COLORS=plt.cm.tab10(np.linspace(0,1,NUM_ACTIONS))
ALPHA_GRID=np.linspace(0.2,1.8,20); PDP_FPS=6; HOLD_FRAMES=12

print('='*70); print('📊 BLOCK 19: PDP + ICE + Validation Improvement'); print('='*70)

# ── 19-A  Validation improvement ─────────────────────────────────────────────
torch.cuda.empty_cache()
def evaluate_val_loss_safe(mdl,n_batches=20):
    mdl.eval(); total,count=0.0,0; loss_fn=nn.CrossEntropyLoss()
    with torch.no_grad():
        for k,(v,inst,tok,tid) in enumerate(loader_val):
            if k>=n_batches: break
            for b in range(v.shape[0]):
                v_s=v[b:b+1].to(DEVICE); tok_s=tok[b:b+1].to(DEVICE)
                emb_s=text_encoder.encode([inst[b]])
                with torch.amp.autocast('cuda',dtype=AUTOCAST_DTYPE):
                    logits=mdl(v_s,emb_s).float()   # (1, CHUNK_SIZE, A, BIN)
                logits_c=logits[:,0,:,:]             # (1, A, BIN) — current step
                B,A,BIN=logits_c.shape
                total+=loss_fn(logits_c.view(B*A,BIN),tok_s.view(B*A).long()).item(); count+=1
                del v_s,tok_s,emb_s,logits
            torch.cuda.empty_cache()
    return total/max(count,1)

print('\n[1/5] Evaluating trained model (n=20 batches)...')
current_val_loss=evaluate_val_loss_safe(model,n_batches=20); torch.cuda.empty_cache()
baseline_val_loss=math.log(256)  # theoretical random
improvement=baseline_val_loss-current_val_loss; pct=improvement/baseline_val_loss*100
improved=improvement>0
print(f'   Baseline (ln 256) : {baseline_val_loss:.4f}')
print(f'   Trained model     : {current_val_loss:.4f}')
print(f'   Δ                 : {improvement:+.4f}  ({pct:+.1f}%)')
print(f'   Result            : {"✅ IMPROVED" if improved else "❌ NOT improved"}')

fig,ax=plt.subplots(figsize=(5,4))
bars=ax.bar(['Random\n(theoretical)','Trained\nModel'],[baseline_val_loss,current_val_loss],
            color=['#e05252','#52e0a0'],width=0.45,edgecolor='black')
for b in bars:
    ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.02,f'{b.get_height():.4f}',
            ha='center',va='bottom',fontsize=11,fontweight='bold')
ax.set_ylabel('Validation Loss (CrossEntropy)'); ax.grid(axis='y',alpha=0.3)
ax.set_title(f'Δ={improvement:+.4f} ({pct:+.1f}%)  {"✅ Improved" if improved else "❌ Not improved"}',fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR19/'validation_improvement.png')

# ── 19-B  PDP + ICE ───────────────────────────────────────────────────────────
print('[2/5] Computing PDP + ICE curves...')
torch.cuda.empty_cache()
def action_score(video,text_emb,alpha):
    v=(video*alpha).clamp(0,1)
    with torch.no_grad(), torch.amp.autocast('cuda',dtype=AUTOCAST_DTYPE):
        logits=model(v,text_emb).float()             # (1, CHUNK_SIZE, A, BIN)
    logits_c=logits[0,0,:,:]                         # (A, BIN) — current step
    bins=torch.arange(256,device=DEVICE).float(); probs=torch.softmax(logits_c,dim=-1)
    score=((probs*bins.unsqueeze(0)).sum(-1)/255.0).cpu().numpy()
    del logits,logits_c,probs; torch.cuda.empty_cache(); return score

pdp_curves=np.zeros((len(ALPHA_GRID),NUM_ACTIONS)); ice_curves=np.zeros((XAI_N,len(ALPHA_GRID),NUM_ACTIONS))
for ai,alpha in enumerate(ALPHA_GRID):
    for si in range(XAI_N):
        score=action_score(xai_videos[si:si+1],xai_text_embs[si:si+1],alpha)
        ice_curves[si,ai]=score; pdp_curves[ai]+=score/XAI_N
    if (ai+1)%5==0: print(f'   α={alpha:.2f}  ({ai+1}/{len(ALPHA_GRID)})')
print('   ✅ done')

# ── 19-C  Static PDP + ICE plot ───────────────────────────────────────────────
print('\n[3/5] Static PDP + ICE plot...')
fig,axes=plt.subplots(2,NUM_ACTIONS,figsize=(4*NUM_ACTIONS,8))
for a in range(NUM_ACTIONS):
    ax=axes[0,a]
    for si in range(XAI_N): ax.plot(ALPHA_GRID,ice_curves[si,:,a],alpha=0.4,lw=1.2,color=ACTION_COLORS[a])
    ax.set_title(f'ICE — {ACTION_NAMES[a]}',fontsize=9,fontweight='bold')
    ax.set_xlabel('α'); ax.set_ylabel('Pred. token'); ax.grid(alpha=0.3)
    ax2=axes[1,a]; ax2.plot(ALPHA_GRID,pdp_curves[:,a],lw=2.5,color='crimson',label='PDP')
    ax2.fill_between(ALPHA_GRID,ice_curves[:,:,a].min(0),ice_curves[:,:,a].max(0),alpha=0.15,color='crimson',label='ICE range')
    ax2.set_title(f'PDP — {ACTION_NAMES[a]}',fontsize=9,fontweight='bold')
    ax2.set_xlabel('α'); ax2.legend(fontsize=7); ax2.grid(alpha=0.3)
plt.suptitle(f'PDP + ICE — Feature: brightness multiplier α  |  {MODEL_NAME}',fontsize=12,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR19/'pdp_ice_static.png')

# ── 19-D  Pre-render animated PDP frames ──────────────────────────────────────
print('\n[4/5] Pre-rendering PDP animation...')
plot_frames=[]; perturbed_left=[]
for ai in range(len(ALPHA_GRID)):
    fig_a,axes_a=plt.subplots(1,NUM_ACTIONS,figsize=(3*NUM_ACTIONS,4),dpi=100)
    for a in range(NUM_ACTIONS):
        ax=axes_a[a]
        for si in range(XAI_N): ax.plot(ALPHA_GRID[:ai+1],ice_curves[si,:ai+1,a],alpha=0.3,lw=1.0,color=ACTION_COLORS[a])
        ax.plot(ALPHA_GRID[:ai+1],pdp_curves[:ai+1,a],lw=2.2,color='crimson')
        ax.set_xlim(ALPHA_GRID[0],ALPHA_GRID[-1])
        ax.set_ylim(ice_curves[:,:,a].min()-0.02,ice_curves[:,:,a].max()+0.02)
        ax.set_title(ACTION_NAMES[a],fontsize=8,fontweight='bold'); ax.set_xlabel('α',fontsize=7)
        if a==0: ax.set_ylabel('Pred. token',fontsize=7)
        ax.grid(alpha=0.25); ax.tick_params(labelsize=6)
    fig_a.suptitle(f'PDP (crimson) + ICE   α={ALPHA_GRID[ai]:.2f}',fontsize=10,fontweight='bold')
    plt.tight_layout(); plot_frames.append(fig_to_rgb(fig_a)); plt.close(fig_a)
    v_p=(xai_videos[0:1]*ALPHA_GRID[ai]).clamp(0,1)
    v_np=v_p[0,:,-1].cpu().numpy().transpose(1,2,0)
    perturbed_left.append(np.clip(v_np,0,1)); del v_p
for _ in range(HOLD_FRAMES):
    plot_frames.append(plot_frames[-1]); perturbed_left.append(perturbed_left[-1])

# ── 19-E  Stitch animation ────────────────────────────────────────────────────
print('\n[5/5] Stitching PDP animation...')
pf_h,pf_w=plot_frames[0].shape[:2]; pw=TARGET_SIZE+40; cw=pw+pf_w
ch=max(TARGET_SIZE+HBAR,pf_h+HBAR); HBAR=42; frames_rgb=[]
for fi in range(len(plot_frames)):
    canvas=np.zeros((ch,cw,3),dtype=np.uint8)
    alpha_val=ALPHA_GRID[min(fi,len(ALPHA_GRID)-1)]
    make_header(canvas,f"PDP/ICE sweep  α={alpha_val:.2f}  crimson=PDP  colour=ICE")
    v_uint8=cv2.resize((perturbed_left[fi]*255).astype(np.uint8),(pw,ch-HBAR))
    canvas[HBAR:,:pw]=v_uint8
    p_rsz=cv2.resize(plot_frames[fi],(pf_w,ch-HBAR))
    canvas[HBAR:,pw:]=p_rsz; frames_rgb.append(canvas)
save_video(frames_rgb,DIR19/'pdp_ice_animation',PDP_FPS)

block_summary(DIR19); print('\n✅ Block 19 done')


📊 BLOCK 19: PDP + ICE + Validation Improvement

[1/5] Evaluating trained model (n=20 batches)...
   Baseline (ln 256) : 5.5452
   Trained model     : 2.2475
   Δ                 : +3.2977  (+59.5%)
   Result            : ✅ IMPROVED
      💾 xai_outputs_mini_vla_mobilevit/block_19_pdp_ice/validation_improvement.png
[2/5] Computing PDP + ICE curves...
   α=0.54  (5/20)
   α=0.96  (10/20)
   α=1.38  (15/20)
   α=1.80  (20/20)
   ✅ done

[3/5] Static PDP + ICE plot...
      💾 xai_outputs_mini_vla_mobilevit/block_19_pdp_ice/pdp_ice_static.png

[4/5] Pre-rendering PDP animation...

[5/5] Stitching PDP animation...
      💾 xai_outputs_mini_vla_mobilevit/block_19_pdp_ice/pdp_ice_animation.gif  (0.8 MB)
      💾 xai_outputs_mini_vla_mobilevit/block_19_pdp_ice/pdp_ice_animation.mp4

   📁 block_19_pdp_ice/
      📄 pdp_ice_animation.gif  808 KB
      📄 pdp_ice_animation.mp4  477 KB
      📄 pdp_ice_static.png  593 KB
      📄 validation_improvement.png  32 KB

✅ Block 19 done


In [20]:
# ========================= BLOCK 20: NOVEL XAI FUSIONS =========================
DIR20=XAI_ROOT/'block_20_fusions'
H_img,W_img=xai_videos.shape[-2],xai_videos.shape[-1]

print('='*70); print('🔬 BLOCK 20: Novel XAI Fusions (ShaGCAM + LimeCAM)'); print('='*70)
torch.cuda.empty_cache(); gc.collect(); torch.cuda.empty_cache()
print(f'   Free VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.2f} GB')

# ── 20-A  Static fusions ──────────────────────────────────────────────────────
print('\n[1/4] Static fusions on XAI_N samples...')
shagcam_results=[]; limecam_results=[]
for i in range(XAI_N):
    torch.cuda.empty_cache()
    frame_np=last_frame_np(xai_videos,i); frame_t=xai_videos[i:i+1,:,-1]
    with GradCAM(model=frame_model,target_layers=target_layer) as gcam:
        gray=gcam(input_tensor=frame_t,targets=cam_targets)[0]
    torch.cuda.empty_cache()
    gcam_map=resize_to(gray,H_img,W_img); gcam_norm=normalise(gcam_map)
    shap_norm=normalise(resize_to(shap_agg_vis[i],H_img,W_img))
    mask_2d=lime_results[i][1]; lime_norm=normalise(resize_to(mask_2d.astype(np.float32),H_img,W_img))
    sha_map=normalise(gcam_norm*shap_norm); sha_ov=show_cam_on_image(frame_np,sha_map,use_rgb=True)
    lc_map=normalise(np.sqrt(gcam_norm*(lime_norm+0.05))); lc_ov=show_cam_on_image(frame_np,lc_map,use_rgb=True)
    shagcam_results.append((frame_np,sha_map,sha_ov)); limecam_results.append((frame_np,lc_map,lc_ov))
    print(f'   sample {i+1}/{XAI_N} done')
torch.cuda.empty_cache()

fig,axes=plt.subplots(XAI_N,5,figsize=(25,5*XAI_N))
col_titles=['Original','GradCAM','|SHAP|','🔬 ShaGCAM\n(SHAP×GradCAM)','🔬 LimeCAM\n(√LIME×GradCAM)']
for j,t in enumerate(col_titles): axes[0,j].set_title(t,fontsize=10,fontweight='bold')
for i in range(XAI_N):
    frame_np=last_frame_np(xai_videos,i); _,_,gcam_ov=cam_results['GradCAM'][i]
    shap_vis=normalise(resize_to(shap_agg_vis[i],H_img,W_img))
    _,_,sha_ov=shagcam_results[i]; _,_,lc_ov=limecam_results[i]
    for col,(img,cmap) in enumerate(zip([frame_np,gcam_ov,shap_vis,sha_ov,lc_ov],[None,None,'hot',None,None])):
        axes[i,col].imshow(img,cmap=cmap); axes[i,col].axis('off')
    axes[i,0].set_ylabel(f'Sample {i+1}',fontsize=9)
plt.suptitle(f'Novel XAI Fusions — {MODEL_NAME}\nShaGCAM=SHAP×GradCAM | LimeCAM=√(LIME×GradCAM)',fontsize=13,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR20/'fusions_static.png')

# ── 20-B  Per-frame GradCAM ───────────────────────────────────────────────────
print(f'\n[2/4] Per-frame GradCAM ({VID_ACTUAL} frames)...')
torch.cuda.empty_cache(); fusion_gcam_maps=[]
with GradCAM(model=frame_model,target_layers=target_layer) as gcam:
    for f_idx in range(VID_ACTUAL):
        gray=gcam(input_tensor=vid_tensors[f_idx],targets=cam_targets)[0]
        fusion_gcam_maps.append(resize_to(gray,H_img,W_img))
        if (f_idx+1)%10==0: torch.cuda.empty_cache()
torch.cuda.empty_cache(); print('   ✅ GradCAM done')

# ── 20-C  Stitch 5-panel fusion video ─────────────────────────────────────────
print('\n[3/4] Stitching fusion video...')
H,W=H_img,W_img; HBAR=45; n_panels=5; cw,ch=W*n_panels,H+HBAR
labels=['Original','GradCAM','|SHAP|','🔬ShaGCAM','🔬LimeCAM']; frames_rgb=[]
for f_idx in range(VID_ACTUAL):
    frame_np=vid_numpys[f_idx]; gcam_norm=normalise(fusion_gcam_maps[f_idx])
    shap_n=(shap_norm_maps[f_idx] if f_idx<len(shap_norm_maps) else np.zeros((H,W),dtype=np.float32))
    shap_rs=normalise(resize_to(shap_n,H,W))
    lime_n=(normalise(resize_to(lime_masks[f_idx].astype(np.float32),H,W)) if f_idx<len(lime_masks) else np.zeros((H,W),dtype=np.float32))
    sha_map=normalise(gcam_norm*shap_rs); sha_ov=show_cam_on_image(frame_np,sha_map,use_rgb=True)
    lc_map=normalise(np.sqrt(gcam_norm*(lime_n+0.05))); lc_ov=show_cam_on_image(frame_np,lc_map,use_rgb=True)
    gcam_ov=show_cam_on_image(frame_np,gcam_norm,use_rgb=True)
    shap_ov=(np.clip(frame_np*0.45+plt.cm.hot(shap_rs)[:,:,:3].astype(np.float32)*0.55,0,1)*255).astype(np.uint8)
    panels=[(frame_np*255).astype(np.uint8),gcam_ov,shap_ov,sha_ov,lc_ov]
    canvas=np.zeros((ch,cw,3),dtype=np.uint8)
    canvas[:HBAR,:]=22
    cv2.putText(canvas,f"XAI Fusions | {f_idx+1}/{VID_ACTUAL} | '{VID_INSTRUCTION}'",(10,HBAR-12),cv2.FONT_HERSHEY_SIMPLEX,0.52,(220,220,220),1)
    for p_idx,(panel,lbl) in enumerate(zip(panels,labels)):
        x0=p_idx*W; canvas[HBAR:,x0:x0+W]=panel
        cv2.putText(canvas,lbl,(x0+4,HBAR+16),cv2.FONT_HERSHEY_SIMPLEX,0.42,(255,255,255),1)
    frames_rgb.append(canvas)
save_video(frames_rgb,DIR20/'fusions_video',VID_FPS)

# ── 20-D  Filmstrip ───────────────────────────────────────────────────────────
print('\n[4/4] Filmstrip...')
strip_idxs=list(range(0,VID_ACTUAL,10)); n_strip=len(strip_idxs)
fig,axes=plt.subplots(3,n_strip,figsize=(3*n_strip,9))
if n_strip==1: axes=axes[:,np.newaxis]
for col,f_idx in enumerate(strip_idxs):
    frame_np=vid_numpys[f_idx]; gcam_norm=normalise(fusion_gcam_maps[f_idx])
    shap_n=(shap_norm_maps[f_idx] if f_idx<len(shap_norm_maps) else np.zeros((H,W)))
    sha_map=normalise(gcam_norm*normalise(resize_to(shap_n,H,W))); sha_ov=show_cam_on_image(frame_np,sha_map,use_rgb=True)
    lime_n=(normalise(resize_to(lime_masks[f_idx].astype(np.float32),H,W)) if f_idx<len(lime_masks) else np.zeros((H,W)))
    lc_map=normalise(np.sqrt(gcam_norm*(lime_n+0.05))); lc_ov=show_cam_on_image(frame_np,lc_map,use_rgb=True)
    axes[0,col].imshow(frame_np); axes[0,col].set_title(f'F{f_idx+1}',fontsize=7); axes[0,col].axis('off')
    if col==0: axes[0,col].set_ylabel('Original',fontsize=9,fontweight='bold')
    axes[1,col].imshow(sha_ov); axes[1,col].axis('off')
    if col==0: axes[1,col].set_ylabel('ShaGCAM',fontsize=9,fontweight='bold')
    axes[2,col].imshow(lc_ov); axes[2,col].axis('off')
    if col==0: axes[2,col].set_ylabel('LimeCAM',fontsize=9,fontweight='bold')
plt.suptitle(f'Novel Fusions Filmstrip — "{VID_INSTRUCTION}"',fontsize=12,fontweight='bold')
plt.tight_layout(); save_fig(fig,DIR20/'fusions_filmstrip.png')

block_summary(DIR20); print('\n✅ Block 20 done')


🔬 BLOCK 20: Novel XAI Fusions (ShaGCAM + LimeCAM)
   Free VRAM: 4.56 GB

[1/4] Static fusions on XAI_N samples...
   sample 1/4 done
   sample 2/4 done
   sample 3/4 done
   sample 4/4 done
      💾 xai_outputs_mini_vla_mobilevit/block_20_fusions/fusions_static.png

[2/4] Per-frame GradCAM (100 frames)...
   ✅ GradCAM done

[3/4] Stitching fusion video...
      💾 xai_outputs_mini_vla_mobilevit/block_20_fusions/fusions_video.gif  (3.9 MB)
      💾 xai_outputs_mini_vla_mobilevit/block_20_fusions/fusions_video.mp4

[4/4] Filmstrip...
      💾 xai_outputs_mini_vla_mobilevit/block_20_fusions/fusions_filmstrip.png

   📁 block_20_fusions/
      📄 fusions_filmstrip.png  5.1 MB
      📄 fusions_static.png  1.5 MB
      📄 fusions_video.gif  3.9 MB
      📄 fusions_video.mp4  2.7 MB

✅ Block 20 done


In [21]:
# ========================= BLOCK 21: XAI MASTER DASHBOARD =========================
DIR21=XAI_ROOT/'block_21_dashboard'
print('='*70); print('🎛️  BLOCK 21: XAI Master Dashboard'); print('='*70)

sample_idx=0; frame_np_s=last_frame_np(xai_videos,sample_idx)
fig=plt.figure(figsize=(26,16)); fig.patch.set_facecolor('#0a0a14')

def dark_ax(ax,title=''):
    ax.set_facecolor('#13132a')
    for sp in ax.spines.values(): sp.set_edgecolor('#3a3a5c')
    ax.tick_params(colors='#aaa',labelsize=7)
    if title: ax.set_title(title,color='#e0e0ff',fontsize=9,fontweight='bold',pad=5)
    ax.axis('off')

gs=fig.add_gridspec(4,7,hspace=0.38,wspace=0.12,top=0.93,bottom=0.04,left=0.02,right=0.98)

# ── Row 0: Title ──────────────────────────────────────────────────────────────
ax_t=fig.add_subplot(gs[0,:]); ax_t.set_facecolor('#0a0a14'); ax_t.axis('off')
ax_t.text(0.5,0.62,f'🔬  XAI Dashboard — {MODEL_NAME}',ha='center',va='center',
          fontsize=18,color='white',fontweight='bold')
ax_t.text(0.5,0.22,f'Task: "{xai_insts[sample_idx]}"   |   Val loss: {current_val_loss:.4f}   |   Improvement vs random: {improvement:+.4f} ({pct:+.1f}%)',
          ha='center',va='center',fontsize=11,color='#aaddff')

# ── Row 1: core XAI methods ───────────────────────────────────────────────────
H_img2,W_img2=xai_videos.shape[-2],xai_videos.shape[-1]
row1_data=[('Original',frame_np_s,None),
           ('GradCAM',cam_results['GradCAM'][sample_idx][2],None),
           ('HiResCAM',cam_results['HiResCAM'][sample_idx][2],None),
           ('ShapleyCAM',cam_results['ShapleyCAM'][sample_idx][2],None),
           ('LIME top-5',lime_results[sample_idx][0],None),
           ('|SHAP|',normalise(resize_to(shap_agg_vis[sample_idx],H_img2,W_img2)),'hot')]
for col,(name,img,cmap) in enumerate(row1_data):
    ax=fig.add_subplot(gs[1,col]); dark_ax(ax,name); ax.imshow(img,cmap=cmap); ax.axis('off')
ax_val=fig.add_subplot(gs[1,6]); dark_ax(ax_val,'Val Loss'); ax_val.axis('on')
bars=ax_val.bar(['Random','Trained'],[baseline_val_loss,current_val_loss],color=['#cc4444','#44cc88'],width=0.5,edgecolor='#aaa')
for b in bars:
    ax_val.text(b.get_x()+b.get_width()/2,b.get_height()+0.01,f'{b.get_height():.3f}',ha='center',va='bottom',color='white',fontsize=9,fontweight='bold')
ax_val.set_facecolor('#13132a'); ax_val.tick_params(colors='#aaa',labelsize=8)
for sp in ax_val.spines.values(): sp.set_edgecolor('#3a3a5c')

# ── Row 2: fusions + evolution ────────────────────────────────────────────────
ax_sha=fig.add_subplot(gs[2,0:2]); dark_ax(ax_sha,'🔬 ShaGCAM  (SHAP × GradCAM)')
ax_sha.imshow(shagcam_results[sample_idx][2]); ax_sha.axis('off')
ax_lc=fig.add_subplot(gs[2,2:4]); dark_ax(ax_lc,'🔬 LimeCAM  (√LIME × GradCAM)')
ax_lc.imshow(limecam_results[sample_idx][2]); ax_lc.axis('off')
frame_np_c=frame_np_s.copy(); _,lmask,_=lime_results[sample_idx]
frame_np_c[lmask==1]=frame_np_c[lmask==1]*0.35+np.array([0.,0.85,0.2])*0.65
ax_lm=fig.add_subplot(gs[2,4]); dark_ax(ax_lm,'LIME Overlay'); ax_lm.imshow(np.clip(frame_np_c,0,1)); ax_lm.axis('off')
shap_n_s=normalise(resize_to(shap_agg_vis[sample_idx],H_img2,W_img2))
shap_b=np.clip(frame_np_s*0.45+plt.cm.hot(shap_n_s)[:,:,:3].astype(np.float32)*0.55,0,1)
ax_sb=fig.add_subplot(gs[2,5]); dark_ax(ax_sb,'SHAP Overlay'); ax_sb.imshow(shap_b); ax_sb.axis('off')
ax_ev=fig.add_subplot(gs[2,6]); dark_ax(ax_ev,'Attn Peak (100 frames)'); ax_ev.axis('on'); ax_ev.set_facecolor('#13132a')
for sp in ax_ev.spines.values(): sp.set_edgecolor('#3a3a5c')
for name in CAM_METHODS:
    ax_ev.plot(np.arange(1,VID_ACTUAL+1),attn_stats[name]['peak'],color=CAM_COLORS[name],lw=1.4,alpha=0.85,label=name)
ax_ev.set_xlabel('Frame',color='#aaa',fontsize=7); ax_ev.set_ylabel('Peak CAM',color='#aaa',fontsize=7)
ax_ev.tick_params(colors='#aaa',labelsize=6); ax_ev.legend(fontsize=6,labelcolor='white',facecolor='#13132a',edgecolor='#3a3a5c')
ax_ev.grid(alpha=0.15,color='#444')

# ── Row 3: PDP + LIME activity + mean SHAP ───────────────────────────────────
ax_pdp=fig.add_subplot(gs[3,0:3]); dark_ax(ax_pdp,'PDP — all actions vs brightness α'); ax_pdp.axis('on'); ax_pdp.set_facecolor('#13132a')
for sp in ax_pdp.spines.values(): sp.set_edgecolor('#3a3a5c')
for a in range(NUM_ACTIONS):
    ax_pdp.plot(ALPHA_GRID,pdp_curves[:,a],label=ACTION_NAMES[a],lw=2.0,color=plt.cm.tab10(a/NUM_ACTIONS))
ax_pdp.set_xlabel('Brightness α',color='#aaa',fontsize=8); ax_pdp.set_ylabel('Pred. token (norm)',color='#aaa',fontsize=8)
ax_pdp.tick_params(colors='#aaa',labelsize=7); ax_pdp.legend(fontsize=7,ncol=4,labelcolor='white',facecolor='#13132a',edgecolor='#3a3a5c')
ax_pdp.grid(alpha=0.15,color='#444')
active_arr_smooth=np.convolve(np.array(active_area),np.ones(10)/10,mode='same')
ax_la=fig.add_subplot(gs[3,3:5]); dark_ax(ax_la,'LIME Active Area (100 frames)'); ax_la.axis('on'); ax_la.set_facecolor('#13132a')
for sp in ax_la.spines.values(): sp.set_edgecolor('#3a3a5c')
ax_la.fill_between(np.arange(1,VID_ACTUAL+1),np.array(active_area),alpha=0.2,color='limegreen')
ax_la.plot(np.arange(1,VID_ACTUAL+1),active_arr_smooth,color='limegreen',lw=2.0)
ax_la.set_xlabel('Frame',color='#aaa',fontsize=8); ax_la.set_ylabel('Fraction highlighted',color='#aaa',fontsize=8)
ax_la.tick_params(colors='#aaa',labelsize=7); ax_la.grid(alpha=0.15,color='#444')
ax_ms=fig.add_subplot(gs[3,5:]); dark_ax(ax_ms,'Mean |SHAP| — 100 frames'); ax_ms.axis('on')
ax_ms.imshow(shap_norm_maps.mean(axis=0),cmap='hot',vmin=0,vmax=1); ax_ms.axis('off')

dash_path=str(DIR21/'xai_dashboard.png')
plt.savefig(dash_path,dpi=180,bbox_inches='tight',facecolor=fig.get_facecolor())
plt.show(); plt.close(fig); print(f'   💾 {dash_path}')

# ── Final tree ────────────────────────────────────────────────────────────────
print(f'\n{"="*70}')
print(f'🎉 ALL XAI BLOCKS COMPLETE — {MODEL_NAME}')
print(f'{"="*70}')
total_kb=0
for sub in sorted(XAI_ROOT.iterdir()):
    if not sub.is_dir(): continue
    sub_kb=sum(f.stat().st_size for f in sub.iterdir())/1024; total_kb+=sub_kb
    print(f'\n   📁 {sub.name}/  ({sub_kb/1024:.1f} MB)')
    for f in sorted(sub.iterdir()):
        kb=f.stat().st_size/1024
        print(f'      📄 {f.name:<45} {"%.1f MB" % (kb/1024) if kb>1024 else "%.0f KB" % kb}')
print(f'\n   Total: {total_kb/1024:.1f} MB  |  Root: {XAI_ROOT.resolve()}')
print(f'{"="*70}')
print("✅ Block 21 done")


🎛️  BLOCK 21: XAI Master Dashboard
   💾 xai_outputs_mini_vla_mobilevit/block_21_dashboard/xai_dashboard.png

🎉 ALL XAI BLOCKS COMPLETE — mini_vla_mobilevit

   📁 block_16_gradcam/  (18.3 MB)
      📄 gradcam_evolution.png                         385 KB
      📄 gradcam_filmstrip.png                         6.9 MB
      📄 gradcam_sidebyside.gif                        3.4 MB
      📄 gradcam_sidebyside.mp4                        2.2 MB
      📄 gradcam_static.png                            5.1 MB
      📄 gradcam_trajectory.png                        346 KB

   📁 block_17_lime/  (8.1 MB)
      📄 lime_activity.png                             684 KB
      📄 lime_filmstrip.png                            3.3 MB
      📄 lime_static.png                               886 KB
      📄 lime_video.gif                                2.0 MB
      📄 lime_video.mp4                                1.3 MB

   📁 block_18_shap/  (7.5 MB)
      📄 shap_filmstrip.png                            3.0 MB
      📄 shap_me